# 03 — Couche Gold

**Étape 3.1a/3.1b : conception du modèle en étoile et vérification des fondations**

Modèle cible (Kimball) :

| Table | Grain |
|---|---|
| `fact_order_items` | une ligne produit d'une commande |
| `fact_orders` | une commande |
| `dim_product` | un produit |
| `dim_seller` | un vendeur |
| `dim_customer` | une personne (`customer_unique_id`) |
| `dim_date` | un jour |

`dim_geo` est abandonnée : `customers` et `sellers` portent déjà `lat`, `lng` et `has_geo` (R12).

Un notebook Spark actif à la fois : fermer le kernel de `02_silver.ipynb` avant de lancer celui-ci.

## 1. Configuration

In [1]:
from pyspark.sql import SparkSession, functions as F, Window

spark = SparkSession.builder.appName("gold").master("local[*]").getOrCreate()

SILVER = "/home/work/data/silver/"
GOLD = "/home/work/data/gold/"

def lire_silver(nom):
    return spark.read.parquet(SILVER + nom)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/09 19:15:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/09 19:15:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## 2. Grain et unicité des clés

In [2]:
oi = lire_silver("order_items")
pr = lire_silver("products")
se = lire_silver("sellers")
cu = lire_silver("customers")
oe = lire_silver("orders_enriched")

print("order_items : lignes                 :", oi.count(), "(attendu 112626)")
print("order_items : couples distincts      :",
      oi.select("order_id", "order_item_id").distinct().count(), "(doit égaler la ligne précédente)")
print("products : lignes / product_id distincts :",
      pr.count(), "/", pr.select("product_id").distinct().count(), "(attendu 32951 / 32951)")
print("sellers  : lignes / seller_id distincts  :",
      se.count(), "/", se.select("seller_id").distinct().count(), "(attendu 3095 / 3095)")
print("customers : lignes / customer_id / customer_unique_id :",
      cu.count(), "/", cu.select("customer_id").distinct().count(), "/",
      cu.select("customer_unique_id").distinct().count(), "(attendu 99441 / 99441 / 96096)")

print("\nOrphelins (attendu 0 partout) :")
print("  order_items -> products :", oi.join(pr, "product_id", "left_anti").count())
print("  order_items -> sellers  :", oi.join(se, "seller_id", "left_anti").count())
print("  order_items -> orders   :", oi.join(oe.select("order_id"), "order_id", "left_anti").count())

order_items : lignes                 : 112626 (attendu 112626)


order_items : couples distincts      : 112626 (doit égaler la ligne précédente)
products : lignes / product_id distincts : 32951 / 32951 (attendu 32951 / 32951)
sellers  : lignes / seller_id distincts  : 3095 / 3095 (attendu 3095 / 3095)
customers : lignes / customer_id / customer_unique_id : 99441 / 99441 / 96096 (attendu 99441 / 99441 / 96096)

Orphelins (attendu 0 partout) :
  order_items -> products : 0
  order_items -> sellers  : 0
  order_items -> orders   : 0


## 3. Ambiguïtés pour `dim_customer` (mesures, sans valeur attendue)

In [3]:
par_personne = (cu.groupBy("customer_unique_id")
    .agg(F.count("*").alias("n_commandes"),
         F.countDistinct("customer_state").alias("n_etats"),
         F.countDistinct("customer_city").alias("n_villes")))

print("Personnes avec plusieurs commandes :", par_personne.filter("n_commandes > 1").count())
print("Personnes avec plusieurs États     :", par_personne.filter("n_etats > 1").count())
print("Personnes avec plusieurs villes    :", par_personne.filter("n_villes > 1").count())

par_zip = (cu.groupBy("customer_zip_code_prefix")
    .agg(F.countDistinct("customer_state").alias("n_etats"),
         F.countDistinct("customer_city").alias("n_villes")))

print("\nCodes postaux clients                 :", par_zip.count())
print("Codes postaux avec plusieurs États    :", par_zip.filter("n_etats > 1").count())
print("Codes postaux avec plusieurs villes   :", par_zip.filter("n_villes > 1").count())

Personnes avec plusieurs commandes : 2997
Personnes avec plusieurs États     : 39
Personnes avec plusieurs villes    : 122

Codes postaux clients                 : 14994
Codes postaux avec plusieurs États    : 0
Codes postaux avec plusieurs villes   : 39


## 4. Bornes de dates et pré-contrôle de la table de faits

In [4]:
o = lire_silver("orders")
o.agg(F.min("order_purchase_timestamp").alias("achat_min"),
      F.max("order_purchase_timestamp").alias("achat_max"),
      F.min("order_estimated_delivery_date").alias("estim_min"),
      F.max("order_estimated_delivery_date").alias("estim_max"),
      F.max("order_delivered_customer_date").alias("livr_max")).show(truncate=False)

base = oi.join(oe.select("order_id", "is_sale", "in_analysis_window", "order_status"),
               "order_id", "left")

print("Lignes après jointure          :", base.count(), "(doit rester 112626)")
print("Lignes sans commande associée  :", base.filter("is_sale IS NULL").count(), "(attendu 0)")
print(f"CA Silver  : {base.agg(F.sum('price')).first()[0]:,.2f} (attendu 13 588 545,09)")
print(f"CA ventes  : {base.filter('is_sale').agg(F.sum('price')).first()[0]:,.2f} (attendu 13 491 302,13)")
print("Lignes produit dans la fenêtre d'analyse :", base.filter("in_analysis_window").count())

+-------------------+-------------------+-------------------+-------------------+-------------------+
|achat_min          |achat_max          |estim_min          |estim_max          |livr_max           |
+-------------------+-------------------+-------------------+-------------------+-------------------+
|2016-09-04 21:15:19|2018-10-17 17:30:18|2016-09-30 00:00:00|2018-11-12 00:00:00|2018-10-17 13:22:46|
+-------------------+-------------------+-------------------+-------------------+-------------------+

Lignes après jointure          : 112626 (doit rester 112626)
Lignes sans commande associée  : 0 (attendu 0)
CA Silver  : 13,588,545.09 (attendu 13 588 545,09)
CA ventes  : 13,491,302.13 (attendu 13 491 302,13)
Lignes produit dans la fenêtre d'analyse : 112255


## 5. Les variantes d'accents expliquent-elles les écarts de villes ?

Mesure sans valeur attendue. Les valeurs « mesuré » viennent de la cellule 3.

In [5]:
def norm_ville(c):
    return F.expr(f"translate(lower(trim({c})), "
                  "'áàâãäéèêëíìîïóòôõöúùûüç', 'aaaaaeeeeiiiiooooouuuuc')")

cu_n = cu.withColumn("ville_norm", norm_ville("customer_city"))

print("Villes distinctes (brut)             :", cu.select("customer_city").distinct().count())
print("Villes distinctes (sans accents)     :", cu_n.select("ville_norm").distinct().count())

pers = cu_n.groupBy("customer_unique_id").agg(
    F.countDistinct("customer_city").alias("n_brut"),
    F.countDistinct("ville_norm").alias("n_norm"))
print("Personnes à plusieurs villes (brut)  :", pers.filter("n_brut > 1").count(), "(mesuré : 122)")
print("Personnes à plusieurs villes (norm.) :", pers.filter("n_norm > 1").count())

zips = cu_n.groupBy("customer_zip_code_prefix").agg(
    F.countDistinct("customer_city").alias("n_brut"),
    F.countDistinct("ville_norm").alias("n_norm"),
    F.collect_set("customer_city").alias("villes"))
print("Codes postaux à plusieurs villes (brut)  :", zips.filter("n_brut > 1").count(), "(mesuré : 39)")
print("Codes postaux à plusieurs villes (norm.) :", zips.filter("n_norm > 1").count())
zips.filter("n_norm > 1").show(5, truncate=False)

Villes distinctes (brut)             : 4119


Villes distinctes (sans accents)     : 4119
Personnes à plusieurs villes (brut)  : 122 (mesuré : 122)


Personnes à plusieurs villes (norm.) : 122
Codes postaux à plusieurs villes (brut)  : 39 (mesuré : 39)
Codes postaux à plusieurs villes (norm.) : 39


[Stage 189:>                                                        (0 + 2) / 2]

+------------------------+------+------+----------------------+
|customer_zip_code_prefix|n_brut|n_norm|villes                |
+------------------------+------+------+----------------------+
|06806                   |2     |2     |[embu das artes, embu]|
|06810                   |2     |2     |[embu das artes, embu]|
|06813                   |2     |2     |[embu das artes, embu]|
|06814                   |2     |2     |[embu das artes, embu]|
|06823                   |2     |2     |[embu das artes, embu]|
+------------------------+------+------+----------------------+
only showing top 5 rows



## 6. `dim_customer` (construite en mémoire, pas encore écrite)

- Grain : une ligne par `customer_unique_id` (96 096 attendues).
- Adresse retenue : celle de la commande la plus récente, `customer_id` en second critère pour un résultat déterministe.
- `fact_orders` gardera l'État de la commande elle-même (région de livraison), qui peut différer de l'adresse actuelle.

In [6]:
o_min = lire_silver("orders").select("order_id", "customer_id", "order_purchase_timestamp")
cu_o = cu.join(o_min, "customer_id", "left")
print("Lignes après jointure clients-commandes :", cu_o.count(), "(attendu 99441, relation 1 pour 1)")

w = Window.partitionBy("customer_unique_id").orderBy(
    F.col("order_purchase_timestamp").desc_nulls_last(), F.col("customer_id").desc())

dim_customer = (cu_o.withColumn("rn", F.row_number().over(w)).filter("rn = 1")
    .select("customer_unique_id", "customer_zip_code_prefix", "customer_city",
            "customer_state", "customer_lat", "customer_lng", "has_geo"))

print("dim_customer : lignes / distincts :", dim_customer.count(), "/",
      dim_customer.select("customer_unique_id").distinct().count(), "(attendu 96096 / 96096)")

# Personnes dont toutes les commandes sont en quarantaine (mesure, sans attendu)
sans_cmd = (cu_o.groupBy("customer_unique_id").agg(F.count("order_id").alias("n"))
            .filter("n = 0").count())
print("Personnes sans aucune commande en Silver :", sans_cmd)

# Vérification visuelle sur les personnes à plusieurs États
multi = (cu.groupBy("customer_unique_id")
           .agg(F.collect_set("customer_state").alias("etats"))
           .filter(F.size("etats") > 1))
(multi.join(dim_customer.select("customer_unique_id", F.col("customer_state").alias("etat_retenu")),
            "customer_unique_id")
      .show(5, truncate=False))

Lignes après jointure clients-commandes : 99441 (attendu 99441, relation 1 pour 1)


dim_customer : lignes / distincts : 96096 / 96096 (attendu 96096 / 96096)
Personnes sans aucune commande en Silver : 29


+--------------------------------+--------+-----------+
|customer_unique_id              |etats   |etat_retenu|
+--------------------------------+--------+-----------+
|547d0504ca415eb4864fa3030f73d3f3|[MG, SP]|MG         |
|5cbfdb85ec130898108b32c50d619c39|[MT, GO]|MT         |
|657dec397f46d84dbe64df2b0389b3cc|[MG, RJ]|RJ         |
|a262442e3ab89611b44877c7aaf77468|[SE, RJ]|RJ         |
|cdc8b0d152b868db7e28de59f642c94a|[SP, PE]|PE         |
+--------------------------------+--------+-----------+
only showing top 5 rows



In [7]:
ex = ["547d0504ca415eb4864fa3030f73d3f3", "5cbfdb85ec130898108b32c50d619c39",
      "657dec397f46d84dbe64df2b0389b3cc", "a262442e3ab89611b44877c7aaf77468",
      "cdc8b0d152b868db7e28de59f642c94a"]

(cu.filter(F.col("customer_unique_id").isin(ex))
   .join(o_min, "customer_id", "left")
   .select("customer_unique_id", "customer_id", "customer_state", "order_purchase_timestamp")
   .orderBy("customer_unique_id", F.col("order_purchase_timestamp").desc_nulls_last())
   .show(20, truncate=False))

non_ascii = (cu.filter(F.col("customer_city").rlike("[^\\x00-\\x7F]"))
               .select("customer_city").distinct().count())
print("Villes contenant un caractère non ASCII :", non_ascii)

+--------------------------------+--------------------------------+--------------+------------------------+
|customer_unique_id              |customer_id                     |customer_state|order_purchase_timestamp|
+--------------------------------+--------------------------------+--------------+------------------------+
|547d0504ca415eb4864fa3030f73d3f3|338a88863afef428379f400aaf88ebb0|MG            |2018-01-21 16:32:51     |
|547d0504ca415eb4864fa3030f73d3f3|9cb217017421d848098b2087a6b97f35|SP            |2017-11-15 07:52:22     |
|5cbfdb85ec130898108b32c50d619c39|2985f6b30228725ea3b233e6c8b87735|MT            |2018-03-14 13:42:40     |
|5cbfdb85ec130898108b32c50d619c39|d9110683c7a282144e9fc97660026a28|GO            |2018-01-24 17:53:44     |
|657dec397f46d84dbe64df2b0389b3cc|ac20409a5bcf2810040956b8fb803c70|RJ            |2017-12-20 15:47:46     |
|657dec397f46d84dbe64df2b0389b3cc|8ee7f235aeeac5ee9f13a4e9739ee290|MG            |2017-11-01 11:29:30     |
|a262442e3ab89611b44877c7aaf

In [9]:
def ecrire_gold(df, nom):
    df.coalesce(1).write.mode("overwrite").parquet(GOLD + nom)

# --- dim_date ---
dim_date = (spark.sql("SELECT explode(sequence(to_date('2016-09-01'), to_date('2018-12-31'), "
                      "interval 1 day)) AS date")
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("year_month", F.date_format("date", "yyyy-MM"))
    .withColumn("day_of_week", F.dayofweek("date"))              # 1 = dimanche
    .withColumn("day_name", F.date_format("date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("date").isin(1, 7))
    .withColumn("in_analysis_window",
                F.col("date").between(F.lit("2017-01-01").cast("date"), F.lit("2018-08-31").cast("date")))
    .select("date_key", "date", "year", "quarter", "month", "year_month",
            "day_of_week", "day_name", "is_weekend", "in_analysis_window"))

# --- dim_product ---
dim_product = (lire_silver("products")
    .withColumnRenamed("product_name_lenght", "product_name_length")
    .withColumnRenamed("product_description_lenght", "product_description_length")
    .withColumn("product_volume_cm3",
                F.col("product_length_cm") * F.col("product_height_cm") * F.col("product_width_cm"))
    .select("product_id", F.col("product_category_name").alias("category_pt"),
            F.col("category_en"), "is_category_unknown",
            "product_name_length", "product_description_length", "product_photos_qty",
            "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
            "product_volume_cm3"))

# --- dim_seller ---
dim_seller = lire_silver("sellers").select(
    "seller_id", "seller_zip_code_prefix", "seller_city", "seller_state",
    "seller_lat", "seller_lng", "has_geo")

# --- dim_customer ---
o_min = lire_silver("orders").select("order_id", "customer_id", "order_purchase_timestamp")
cu_o = lire_silver("customers").join(o_min, "customer_id", "inner")
print("Lignes clients-commandes (inner) :", cu_o.count(), "(attendu 99410)")

w = Window.partitionBy("customer_unique_id").orderBy(
    F.col("order_purchase_timestamp").desc(), F.col("customer_id").desc())

dim_customer = (cu_o.withColumn("rn", F.row_number().over(w)).filter("rn = 1")
    .select("customer_unique_id", "customer_zip_code_prefix", "customer_city",
            "customer_state", "customer_lat", "customer_lng", "has_geo"))

for nom, df in [("dim_date", dim_date), ("dim_product", dim_product),
                ("dim_seller", dim_seller), ("dim_customer", dim_customer)]:
    ecrire_gold(df, nom)
print("Écrit : dim_date, dim_product, dim_seller, dim_customer")

Lignes clients-commandes (inner) : 99410 (attendu 99410)


Écrit : dim_date, dim_product, dim_seller, dim_customer


In [10]:
attendu = {"dim_date": 852, "dim_product": 32951, "dim_seller": 3095, "dim_customer": 96067}
for nom, n_att in attendu.items():
    d = spark.read.parquet(GOLD + nom)
    print(f"{nom:14} lignes={d.count():>7} (attendu {n_att})  colonnes={len(d.columns)}")

dd = spark.read.parquet(GOLD + "dim_date")
dp = spark.read.parquet(GOLD + "dim_product")
ds = spark.read.parquet(GOLD + "dim_seller")
dc = spark.read.parquet(GOLD + "dim_customer")

print("\ndim_date")
print("  date_key distincts      :", dd.select("date_key").distinct().count(), "(attendu 852)")
print("  jours dans la fenêtre   :", dd.filter("in_analysis_window").count(), "(attendu 608)")
dd.agg(F.min("date"), F.max("date")).show()
o_keys = lire_silver("orders").select(
    F.date_format("order_purchase_timestamp", "yyyyMMdd").cast("int").alias("date_key"))
print("  commandes sans date dans dim_date :", o_keys.join(dd, "date_key", "left_anti").count(), "(attendu 0)")

print("\ndim_product")
print("  product_id distincts    :", dp.select("product_id").distinct().count(), "(attendu 32951)")
print("  catégorie unknown       :", dp.filter("is_category_unknown").count(), "(attendu 610)")
print("  poids NULL              :", dp.filter("product_weight_g IS NULL").count(), "(attendu 6)")
print("  category_en NULL        :", dp.filter("category_en IS NULL").count(), "(attendu 0)")

print("\ndim_seller")
print("  seller_id distincts     :", ds.select("seller_id").distinct().count(), "(attendu 3095)")
print("  sans géolocalisation    :", ds.filter("NOT has_geo").count(), "(attendu 7)")

print("\ndim_customer")
print("  customer_unique_id distincts :", dc.select("customer_unique_id").distinct().count(), "(attendu 96067)")
print("  sans géolocalisation (mesure) :", dc.filter("NOT has_geo").count())

dim_date       lignes=    852 (attendu 852)  colonnes=10
dim_product    lignes=  32951 (attendu 32951)  colonnes=12
dim_seller     lignes=   3095 (attendu 3095)  colonnes=7
dim_customer   lignes=  96067 (attendu 96067)  colonnes=7

dim_date
  date_key distincts      : 852 (attendu 852)
  jours dans la fenêtre   : 608 (attendu 608)
+----------+----------+
| min(date)| max(date)|
+----------+----------+
|2016-09-01|2018-12-31|
+----------+----------+

  commandes sans date dans dim_date : 0 (attendu 0)

dim_product
  product_id distincts    : 32951 (attendu 32951)
  catégorie unknown       : 610 (attendu 610)
  poids NULL              : 6 (attendu 6)
  category_en NULL        : 0 (attendu 0)

dim_seller
  seller_id distincts     : 3095 (attendu 3095)
  sans géolocalisation    : 7 (attendu 7)

dim_customer
  customer_unique_id distincts : 96067 (attendu 96067)
  sans géolocalisation (mesure) : 268


In [11]:
oi = lire_silver("order_items")
oe = lire_silver("orders_enriched")

order_date_key = F.date_format("order_purchase_timestamp", "yyyyMMdd").cast("int")

# --- fact_order_items ---
entete = (oe.select("order_id", "customer_unique_id", "customer_state", "order_status",
                    "is_sale", "in_analysis_window",
                    order_date_key.alias("order_date_key")))

fact_order_items = (oi.select("order_id", "order_item_id", "product_id", "seller_id",
                              "shipping_limit_date", "price", "freight_value")
                      .join(entete, "order_id", "inner"))

# --- fact_orders ---
a_retirer = ["customer_id", "customer_city", "customer_has_geo",
             "review_creation_date", "review_answer_timestamp",
             "ingestion_ts", "source_file", "batch_id"]
fact_orders = oe.withColumn("order_date_key", order_date_key).drop(*a_retirer)

print("fact_order_items :", fact_order_items.count(), "lignes,", len(fact_order_items.columns), "colonnes (attendu 112626 / 13)")
print("fact_orders      :", fact_orders.count(), "lignes,", len(fact_orders.columns), "colonnes (attendu 99410 / 34)")

for nom, df in [("fact_order_items", fact_order_items), ("fact_orders", fact_orders)]:
    ecrire_gold(df, nom)
print("Écrit : fact_order_items, fact_orders")

fact_order_items : 112626 lignes, 13 colonnes (attendu 112626 / 13)
fact_orders      : 99410 lignes, 34 colonnes (attendu 99410 / 34)


26/11/09 19:21:24 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Écrit : fact_order_items, fact_orders


In [12]:
foi = spark.read.parquet(GOLD + "fact_order_items")
fo = spark.read.parquet(GOLD + "fact_orders")
dp = spark.read.parquet(GOLD + "dim_product")
ds = spark.read.parquet(GOLD + "dim_seller")
dc = spark.read.parquet(GOLD + "dim_customer")
dd = spark.read.parquet(GOLD + "dim_date")

print("fact_order_items")
print("  lignes                    :", foi.count(), "(attendu 112626)")
print("  couples (order, item)     :", foi.select("order_id", "order_item_id").distinct().count(), "(attendu 112626)")
print(f"  CA total                  : {foi.agg(F.sum('price')).first()[0]:,.2f} (attendu 13 588 545,09)")
print(f"  CA des ventes (is_sale)   : {foi.filter('is_sale').agg(F.sum('price')).first()[0]:,.2f} (attendu 13 491 302,13)")
print("  orphelins -> dim_product  :", foi.join(dp, "product_id", "left_anti").count(), "(attendu 0)")
print("  orphelins -> dim_seller   :", foi.join(ds, "seller_id", "left_anti").count(), "(attendu 0)")
print("  orphelins -> dim_customer :", foi.join(dc, "customer_unique_id", "left_anti").count(), "(attendu 0)")
print("  orphelins -> dim_date     :", foi.join(dd, "order_date_key" if False else foi.order_date_key == dd.date_key, "left_anti").count(), "(attendu 0)")

print("\nfact_orders")
print("  lignes / order_id distincts :", fo.count(), "/", fo.select("order_id").distinct().count(), "(attendu 99410 / 99410)")
print("  somme de n_items          :", fo.agg(F.sum("n_items")).first()[0], "(attendu 112626)")
print(f"  somme de items_total      : {fo.agg(F.sum('items_total')).first()[0]:,.2f} (attendu 13 588 545,09)")
print(f"  items_total (is_sale)     : {fo.filter('is_sale').agg(F.sum('items_total')).first()[0]:,.2f} (attendu 13 491 302,13)")
print("  orphelins -> dim_customer :", fo.join(dc, "customer_unique_id", "left_anti").count(), "(attendu 0)")
print("  orphelins -> dim_date     :", fo.join(dd, fo.order_date_key == dd.date_key, "left_anti").count(), "(attendu 0)")

# Cohérence entre les deux faits : par commande, somme des prix = items_total, lignes = n_items
chk = (foi.groupBy("order_id").agg(F.sum("price").alias("p"), F.count("*").alias("n"))
          .join(fo.select("order_id", "items_total", "n_items"), "order_id", "full_outer")
          .fillna({"p": 0.0, "n": 0}))
ecarts = chk.filter((F.abs(F.col("p") - F.col("items_total")) > 0.01) | (F.col("n") != F.col("n_items")))
print("\nCommandes dont les deux faits ne concordent pas :", ecarts.count(), "(attendu 0)")

fact_order_items
  lignes                    : 112626 (attendu 112626)
  couples (order, item)     : 112626 (attendu 112626)
  CA total                  : 13,588,545.09 (attendu 13 588 545,09)
  CA des ventes (is_sale)   : 13,491,302.13 (attendu 13 491 302,13)
  orphelins -> dim_product  : 0 (attendu 0)
  orphelins -> dim_seller   : 0 (attendu 0)
  orphelins -> dim_customer : 0 (attendu 0)
  orphelins -> dim_date     : 0 (attendu 0)

fact_orders
  lignes / order_id distincts : 99410 / 99410 (attendu 99410 / 99410)
  somme de n_items          : 112626 (attendu 112626)
  somme de items_total      : 13,588,545.09 (attendu 13 588 545,09)
  items_total (is_sale)     : 13,491,302.13 (attendu 13 491 302,13)
  orphelins -> dim_customer : 0 (attendu 0)
  orphelins -> dim_date     : 0 (attendu 0)


[Stage 391:======================================>                  (2 + 1) / 3]


Commandes dont les deux faits ne concordent pas : 0 (attendu 0)


In [13]:
foi = spark.read.parquet(GOLD + "fact_order_items")
dp = spark.read.parquet(GOLD + "dim_product")
dd = spark.read.parquet(GOLD + "dim_date")

ventes = (foi.filter("is_sale")
    .join(F.broadcast(dp.select("product_id", "category_en", "is_category_unknown")),
          "product_id", "left")
    .join(F.broadcast(dd.select(F.col("date_key").alias("order_date_key"), "year_month")),
          "order_date_key", "left"))

print("Lignes de vente              :", ventes.count())
print("Lignes is_sale du fait       :", foi.filter("is_sale").count(), "(doit être égal : jointures sans fan-out)")
print("Sans catégorie (NULL)        :", ventes.filter("category_en IS NULL").count(), "(attendu 0)")
print("Sans year_month (NULL)       :", ventes.filter("year_month IS NULL").count(), "(attendu 0)")
print(f"CA des ventes                : {ventes.agg(F.sum('price')).first()[0]:,.2f} (attendu 13 491 302,13)")

Lignes de vente              : 112077
Lignes is_sale du fait       : 112077 (doit être égal : jointures sans fan-out)
Sans catégorie (NULL)        : 0 (attendu 0)
Sans year_month (NULL)       : 0 (attendu 0)
CA des ventes                : 13,491,302.13 (attendu 13 491 302,13)


In [14]:
ca_total = ventes.agg(F.sum("price")).first()[0]

par_cat = (ventes.groupBy("category_en")
    .agg(F.round(F.sum("price"), 2).alias("ca"),
         F.count("*").alias("n_lignes"),
         F.countDistinct("order_id").alias("n_commandes")))

ordre = Window.orderBy(F.col("ca").desc(), F.col("category_en"))
cumul = ordre.rowsBetween(Window.unboundedPreceding, Window.currentRow)

pareto = (par_cat
    .withColumn("part", F.col("ca") / F.lit(ca_total))
    .withColumn("rang", F.row_number().over(ordre))
    .withColumn("part_cumulee", F.sum("part").over(cumul))
    .withColumn("dans_les_80", (F.col("part_cumulee") - F.col("part")) < 0.8))

print("Catégories avec ventes                   :", pareto.count())
print(f"Somme du CA par catégorie                : {pareto.agg(F.sum('ca')).first()[0]:,.2f} (attendu 13 491 302,13)")
print("Catégories pour atteindre 80 % du CA     :", pareto.filter("dans_les_80").count())
print("Part cumulée de la dernière catégorie    :",
      round(pareto.agg(F.max('part_cumulee')).first()[0], 6), "(attendu 1.0)")

(pareto.orderBy("rang")
    .select("rang", "category_en", "ca", "n_commandes",
            F.round("part", 4).alias("part"), F.round("part_cumulee", 4).alias("cumul"),
            "dans_les_80")
    .show(15, truncate=False))

(pareto.filter("category_en = 'unknown'")
    .select("rang", "category_en", "ca", F.round("part", 4).alias("part")).show())

Catégories avec ventes                   : 74
Somme du CA par catégorie                : 13,491,302.13 (attendu 13 491 302,13)


26/11/09 19:22:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:22:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:22:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:22:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:22:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:22:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 1

Catégories pour atteindre 80 % du CA     : 18


26/11/09 19:22:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:22:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:22:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:22:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:22:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:22:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 1

Part cumulée de la dernière catégorie    : 1.0 (attendu 1.0)


26/11/09 19:22:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:22:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:22:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:22:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:22:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:22:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 1

+----+---------------------+----------+-----------+------+------+-----------+
|rang|category_en          |ca        |n_commandes|part  |cumul |dans_les_80|
+----+---------------------+----------+-----------+------+------+-----------+
|1   |health_beauty        |1255695.13|8800       |0.0931|0.0931|true       |
|2   |watches_gifts        |1197907.21|5602       |0.0888|0.1819|true       |
|3   |bed_bath_table       |1035830.07|9398       |0.0768|0.2586|true       |
|4   |sports_leisure       |979492.94 |7670       |0.0726|0.3312|true       |
|5   |computers_accessories|904211.03 |6653       |0.067 |0.3983|true       |
|6   |furniture_decor      |727030.45 |6421       |0.0539|0.4522|true       |
|7   |housewares           |626825.8  |5847       |0.0465|0.4986|true       |
|8   |cool_stuff           |620530.52 |3613       |0.046 |0.5446|true       |
|9   |auto                 |586468.43 |3871       |0.0435|0.5881|true       |
|10  |garden_tools         |480874.94 |3504       |0.0356|0.6237

26/11/09 19:22:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:22:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:22:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:22:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:22:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:22:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 1

+----+-----------+---------+------+
|rang|category_en|       ca|  part|
+----+-----------+---------+------+
|  21|    unknown|178522.56|0.0132|
+----+-----------+---------+------+



In [15]:
fen = ventes.filter("in_analysis_window")
ca_fenetre = fen.agg(F.sum("price")).first()[0]
print(f"CA des ventes dans la fenêtre : {ca_fenetre:,.2f}")
print(f"CA des ventes hors fenêtre    : {ca_total - ca_fenetre:,.2f}  (mesure : total - fenêtre)")

# --- Évolution mensuelle par catégorie ---
mensuel = (fen.groupBy("year_month", "category_en")
    .agg(F.round(F.sum("price"), 2).alias("ca"),
         F.countDistinct("order_id").alias("n_commandes"))
    .withColumn("ca_mois", F.sum("ca").over(Window.partitionBy("year_month")))
    .withColumn("part_dans_mois", F.col("ca") / F.col("ca_mois")))

mois = mensuel.select("year_month").distinct()
print("Mois distincts                :", mois.count(), "(attendu 20)")
print("Premier / dernier mois        :", mois.agg(F.min("year_month"), F.max("year_month")).first())

# Part des 5 premières catégories (classement toute période) dans le CA de chaque mois
top5 = [r["category_en"] for r in pareto.orderBy("rang").limit(5).collect()]
print("Top 5 :", top5)
(mensuel.filter(F.col("category_en").isin(top5))
    .groupBy("year_month").pivot("category_en", top5)
    .agg(F.round(F.first("part_dans_mois"), 4))
    .orderBy("year_month").show(20, truncate=False))

# --- Le classement change-t-il si on se limite à la fenêtre ? ---
rang_fen = (fen.groupBy("category_en").agg(F.round(F.sum("price"), 2).alias("ca_fenetre"))
    .withColumn("rang_fenetre",
                F.row_number().over(Window.orderBy(F.col("ca_fenetre").desc(), "category_en"))))

comp = (pareto.select("category_en", "rang")
    .join(rang_fen.select("category_en", "rang_fenetre"), "category_en", "full_outer")
    .withColumn("ecart", F.coalesce(F.col("rang") - F.col("rang_fenetre"), F.lit(999))))

print("Catégories sans vente dans la fenêtre :", comp.filter("rang_fenetre IS NULL").count())
print("Catégories dont le rang change        :", comp.filter("ecart != 0").count(), "sur", comp.count())
print("Top 10 : rang toute période -> rang fenêtre")
(comp.filter("rang <= 10 OR rang_fenetre <= 10")
     .orderBy("rang").show(15, truncate=False))

CA des ventes dans la fenêtre : 13,446,431.07
CA des ventes hors fenêtre    : 44,871.06  (mesure : total - fenêtre)
Mois distincts                : 20 (attendu 20)
Premier / dernier mois        : Row(min(year_month)='2017-01', max(year_month)='2018-08')


26/11/09 19:24:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 1

Top 5 : ['health_beauty', 'watches_gifts', 'bed_bath_table', 'sports_leisure', 'computers_accessories']
+----------+-------------+-------------+--------------+--------------+---------------------+
|year_month|health_beauty|watches_gifts|bed_bath_table|sports_leisure|computers_accessories|
+----------+-------------+-------------+--------------+--------------+---------------------+
|2017-01   |0.1047       |0.0674       |0.033         |0.0806        |0.0327               |
|2017-02   |0.0938       |0.0483       |0.066         |0.0665        |0.0492               |
|2017-03   |0.0704       |0.0678       |0.07          |0.0729        |0.0764               |
|2017-04   |0.0644       |0.0663       |0.0688        |0.0687        |0.0553               |
|2017-05   |0.093        |0.0752       |0.0662        |0.0708        |0.0903               |
|2017-06   |0.0744       |0.0661       |0.0817        |0.0761        |0.0861               |
|2017-07   |0.0706       |0.0723       |0.1297        |0.07

26/11/09 19:24:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 1

Catégories sans vente dans la fenêtre : 0


26/11/09 19:24:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 1

Catégories dont le rang change        : 4 sur 74
Top 10 : rang toute période -> rang fenêtre


26/11/09 19:24:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 1

+---------------------+----+------------+-----+
|category_en          |rang|rang_fenetre|ecart|
+---------------------+----+------------+-----+
|health_beauty        |1   |1           |0    |
|watches_gifts        |2   |2           |0    |
|bed_bath_table       |3   |3           |0    |
|sports_leisure       |4   |4           |0    |
|computers_accessories|5   |5           |0    |
|furniture_decor      |6   |6           |0    |
|housewares           |7   |7           |0    |
|cool_stuff           |8   |8           |0    |
|auto                 |9   |9           |0    |
|garden_tools         |10  |10          |0    |
+---------------------+----+------------+-----+



In [16]:
pareto_out = (pareto.withColumn("perimetre", F.lit("toutes_ventes"))
    .select("rang", "category_en", "ca", "n_lignes", "n_commandes",
            "part", "part_cumulee", "dans_les_80", "perimetre"))

mensuel_out = (mensuel.withColumn("perimetre", F.lit("fenetre_2017-01_2018-08"))
    .select("year_month", "category_en", "ca", "n_commandes",
            "ca_mois", "part_dans_mois", "perimetre"))

ecrire_gold(pareto_out, "kpi_m1_pareto")
ecrire_gold(mensuel_out, "kpi_m1_mensuel")

p = spark.read.parquet(GOLD + "kpi_m1_pareto")
m = spark.read.parquet(GOLD + "kpi_m1_mensuel")

print("kpi_m1_pareto  : lignes", p.count(), "(attendu 74)")
print(f"  CA total      : {p.agg(F.sum('ca')).first()[0]:,.2f} (attendu 13 491 302,13)")
print("  catégories dans les 80 % :", p.filter("dans_les_80").count(), "(attendu 18)")

print("kpi_m1_mensuel : lignes", m.count(), "(mesure)")
print("  mois distincts :", m.select("year_month").distinct().count(), "(attendu 20)")
ecart_ca = abs(m.agg(F.sum("ca")).first()[0] - ca_fenetre)
print(f"  écart entre CA relu et CA de la fenêtre : {ecart_ca:.4f} (attendu 0.0000)")

26/11/09 19:24:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 19:24:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 1

kpi_m1_pareto  : lignes 74 (attendu 74)
  CA total      : 13,491,302.13 (attendu 13 491 302,13)
  catégories dans les 80 % : 18 (attendu 18)
kpi_m1_mensuel : lignes 1245 (mesure)
  mois distincts : 20 (attendu 20)
  écart entre CA relu et CA de la fenêtre : 0.0000 (attendu 0.0000)


In [17]:
fo = spark.read.parquet(GOLD + "fact_orders")
livrees = fo.filter("order_status = 'delivered'")

print("Commandes delivered en Silver       :", livrees.count(), "(attendu 96455 = 96478 - 23 en quarantaine)")
print("  sans delivery_days (NULL)         :", livrees.filter("delivery_days IS NULL").count(), "(attendu 0)")
print("  delivery_days < 0                 :", livrees.filter("delivery_days < 0").count(), "(attendu 0, cf. audit)")
print("  dans la fenêtre d'analyse         :", livrees.filter("in_analysis_window").count(), "(mesure)")
print("  en retard (is_late), toute période:", livrees.filter("is_late").count(), "(mesure ; au plus 6534)")
print("  is_late différent de delay_days > 0 :",
      livrees.filter(F.col("is_late") != (F.col("delay_days") > 0)).count(), "(attendu 0 si la définition est celle-là)")

(livrees.filter("in_analysis_window")
    .agg(F.round(F.avg("delivery_days"), 2).alias("moyenne"),
         F.round(F.expr("percentile_approx(delivery_days, 0.5)"), 2).alias("mediane"),
         F.round(F.expr("percentile_approx(delivery_days, 0.9)"), 2).alias("p90"),
         F.round(F.expr("percentile_approx(delivery_days, 0.99)"), 2).alias("p99"),
         F.round(F.max("delivery_days"), 2).alias("max"))
    .show())

Commandes delivered en Silver       : 96455 (attendu 96455 = 96478 - 23 en quarantaine)
  sans delivery_days (NULL)         : 0 (attendu 0)
  delivery_days < 0                 : 0 (attendu 0, cf. audit)
  dans la fenêtre d'analyse         : 96188 (mesure)
  en retard (is_late), toute période: 6533 (mesure ; au plus 6534)
  is_late différent de delay_days > 0 : 0 (attendu 0 si la définition est celle-là)
+-------+-------+-----+----+------+
|moyenne|mediane|  p90| p99|   max|
+-------+-------+-----+----+------+
|  12.54|  10.21|23.06|46.0|209.63|
+-------+-------+-----+----+------+



In [18]:
m2 = (livrees.filter("in_analysis_window")
    .groupBy("customer_state")
    .agg(F.count("*").alias("n_commandes"),
         F.round(F.avg("delivery_days"), 2).alias("delai_moyen"),
         F.round(F.expr("percentile_approx(delivery_days, 0.5)"), 2).alias("delai_median"),
         F.round(F.expr("percentile_approx(delivery_days, 0.9)"), 2).alias("delai_p90"),
         F.round(F.avg(F.col("is_late").cast("int")), 4).alias("taux_retard"),
         F.round(F.avg("delay_days"), 2).alias("ecart_moyen_vs_promis_j")))

print("États :", m2.count(), "(mesure)")
print("Somme des n_commandes :", m2.agg(F.sum("n_commandes")).first()[0],
      "(doit égaler la ligne « dans la fenêtre d'analyse » de la cellule 16)")
m2.orderBy(F.desc("delai_moyen")).show(30, truncate=False)

États : 27 (mesure)
Somme des n_commandes : 96188 (doit égaler la ligne « dans la fenêtre d'analyse » de la cellule 16)
+--------------+-----------+-----------+------------+---------+-----------+-----------------------+
|customer_state|n_commandes|delai_moyen|delai_median|delai_p90|taux_retard|ecart_moyen_vs_promis_j|
+--------------+-----------+-----------+------------+---------+-----------+-----------------------+
|RR            |40         |29.91      |25.01       |53.19    |0.125      |-16.28                 |
|AP            |67         |27.19      |24.35       |35.16    |0.0299     |-19.69                 |
|AM            |145        |26.43      |25.88       |39.17    |0.0276     |-19.57                 |
|AL            |396        |24.52      |22.33       |40.42    |0.2146     |-8.64                  |
|PA            |942        |23.76      |21.06       |39.36    |0.1125     |-13.96                 |
|MA            |712        |21.55      |19.18       |34.29    |0.1756     |-9.39

In [19]:
Z = 1.96

REGIONS = {
    "AC": "Norte", "AP": "Norte", "AM": "Norte", "PA": "Norte", "RO": "Norte", "RR": "Norte", "TO": "Norte",
    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste", "MA": "Nordeste", "PB": "Nordeste",
    "PE": "Nordeste", "PI": "Nordeste", "RN": "Nordeste", "SE": "Nordeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MT": "Centro-Oeste", "MS": "Centro-Oeste",
    "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
    "PR": "Sul", "RS": "Sul", "SC": "Sul"}
region_map = F.create_map(*[x for kv in REGIONS.items() for x in (F.lit(kv[0]), F.lit(kv[1]))])

base_m2 = (livrees.filter("in_analysis_window")
    .withColumn("region", region_map[F.col("customer_state")])
    .withColumn("delai_promis_j",
                (F.unix_timestamp("order_estimated_delivery_date")
                 - F.unix_timestamp("order_purchase_timestamp")) / 86400)
    .withColumn("national", F.lit("BR")))

def kpi_livraison(df, cle):
    agg = df.groupBy(cle).agg(
        F.count("*").alias("n_commandes"),
        F.round(F.avg("delivery_days"), 2).alias("delai_moyen"),
        F.round(F.expr("percentile_approx(delivery_days, 0.5)"), 2).alias("delai_median"),
        F.round(F.expr("percentile_approx(delivery_days, 0.9)"), 2).alias("delai_p90"),
        F.round(F.avg("delai_promis_j"), 2).alias("delai_promis_moyen"),
        F.avg(F.col("is_late").cast("int")).alias("taux_retard"))
    n, p = F.col("n_commandes"), F.col("taux_retard")
    d = 1 + (Z ** 2) / n
    centre = (p + (Z ** 2) / (2 * n)) / d
    marge = Z * F.sqrt(p * (1 - p) / n + (Z ** 2) / (4 * n * n)) / d
    return (agg.withColumn("ic95_bas", F.round(centre - marge, 4))
               .withColumn("ic95_haut", F.round(centre + marge, 4))
               .withColumn("taux_retard", F.round(p, 4)))

m2_etat = kpi_livraison(base_m2, "customer_state").withColumn("region", region_map[F.col("customer_state")])
m2_region = kpi_livraison(base_m2, "region")
m2_national = kpi_livraison(base_m2, "national")

print("Commandes sans région (NULL)  :", base_m2.filter("region IS NULL").count(), "(attendu 0)")
print("Somme n États / régions       :", m2_etat.agg(F.sum("n_commandes")).first()[0], "/",
      m2_region.agg(F.sum("n_commandes")).first()[0], "(attendu 96188 des deux côtés)")

print("\nNational")
m2_national.show(truncate=False)
print("Par région")
m2_region.orderBy("delai_moyen").show(truncate=False)
print("Par État, du plus fort taux de retard au plus faible")
(m2_etat.select("customer_state", "region", "n_commandes", "delai_moyen", "delai_promis_moyen",
                "taux_retard", "ic95_bas", "ic95_haut")
    .orderBy(F.desc("taux_retard")).show(27, truncate=False))

Commandes sans région (NULL)  : 0 (attendu 0)
Somme n États / régions       : 96188 / 96188 (attendu 96188 des deux côtés)

National
+--------+-----------+-----------+------------+---------+------------------+-----------+--------+---------+
|national|n_commandes|delai_moyen|delai_median|delai_p90|delai_promis_moyen|taux_retard|ic95_bas|ic95_haut|
+--------+-----------+-----------+------------+---------+------------------+-----------+--------+---------+
|BR      |96188      |12.54      |10.21       |23.06    |23.65             |0.0679     |0.0663  |0.0695   |
+--------+-----------+-----------+------------+---------+------------------+-----------+--------+---------+

Par région
+------------+-----------+-----------+------------+---------+------------------+-----------+--------+---------+
|region      |n_commandes|delai_moyen|delai_median|delai_p90|delai_promis_moyen|taux_retard|ic95_bas|ic95_haut|
+------------+-----------+-----------+------------+---------+------------------+-----------

In [20]:
ecrire_gold(m2_etat, "kpi_m2_etat")
ecrire_gold(m2_region, "kpi_m2_region")

e = spark.read.parquet(GOLD + "kpi_m2_etat")
r = spark.read.parquet(GOLD + "kpi_m2_region")
print("kpi_m2_etat   : lignes", e.count(), "(attendu 27), n =", e.agg(F.sum("n_commandes")).first()[0], "(attendu 96188)")
print("kpi_m2_region : lignes", r.count(), "(attendu 5),  n =", r.agg(F.sum("n_commandes")).first()[0], "(attendu 96188)")
print("IC incohérents (bas > taux ou haut < taux) :",
      e.filter("ic95_bas > taux_retard OR ic95_haut < taux_retard").count(), "(attendu 0)")

kpi_m2_etat   : lignes 27 (attendu 27), n = 96188 (attendu 96188)
kpi_m2_region : lignes 5 (attendu 5),  n = 96188 (attendu 96188)
IC incohérents (bas > taux ou haut < taux) : 0 (attendu 0)


In [21]:
m3_base = (livrees.filter("in_analysis_window AND has_review")
    .withColumn("tranche_retard",
        F.when(F.col("delay_days") <= 0, "0_a_l_heure_ou_en_avance")
         .when(F.col("delay_days") <= 3, "1_retard_1_3_j")
         .when(F.col("delay_days") <= 7, "2_retard_4_7_j")
         .when(F.col("delay_days") <= 14, "3_retard_8_14_j")
         .otherwise("4_retard_plus_14_j")))

n_fenetre = livrees.filter("in_analysis_window").count()
n_avis = m3_base.count()
print("Livrées dans la fenêtre         :", n_fenetre, "(attendu 96188)")
print("  avec un avis                  :", n_avis, "(mesure)")
print("  sans avis                     :", n_fenetre - n_avis, "(mesure)")
print("  review_score hors 1..5        :", m3_base.filter("review_score NOT BETWEEN 1 AND 5").count(), "(attendu 0)")
print("  tranche NULL                  :", m3_base.filter("tranche_retard IS NULL").count(), "(attendu 0)")
print("  is_late / tranches de retard  :", m3_base.filter("is_late").count(), "/",
      m3_base.filter("tranche_retard != '0_a_l_heure_ou_en_avance'").count(), "(doivent être égaux)")
print("  corrélation delay_days / note :", round(m3_base.stat.corr("delay_days", "review_score"), 4), "(mesure)")
m3_base.groupBy("tranche_retard").count().orderBy("tranche_retard").show(truncate=False)

Livrées dans la fenêtre         : 96188 (attendu 96188)
  avec un avis                  : 95545 (mesure)
  sans avis                     : 643 (mesure)
  review_score hors 1..5        : 0 (attendu 0)
  tranche NULL                  : 0 (attendu 0)
  is_late / tranches de retard  : 6377 / 6377 (doivent être égaux)
  corrélation delay_days / note : -0.2704 (mesure)
+------------------------+-----+
|tranche_retard          |count|
+------------------------+-----+
|0_a_l_heure_ou_en_avance|89168|
|1_retard_1_3_j          |1851 |
|2_retard_4_7_j          |1747 |
|3_retard_8_14_j         |1446 |
|4_retard_plus_14_j      |1333 |
+------------------------+-----+



In [22]:
def kpi_avis(df, cle):
    agg = df.groupBy(cle).agg(
        F.count("*").alias("n_avis"),
        F.avg("review_score").alias("note_moyenne"),
        F.stddev("review_score").alias("ecart_type"),
        F.avg((F.col("review_score") <= 2).cast("int")).alias("part_notes_1_2"),
        F.avg((F.col("review_score") == 5).cast("int")).alias("part_notes_5"))
    return (agg.withColumn("marge95_note", F.round(Z * F.col("ecart_type") / F.sqrt("n_avis"), 3))
               .withColumn("note_moyenne", F.round("note_moyenne", 3))
               .withColumn("ecart_type", F.round("ecart_type", 3))
               .withColumn("part_notes_1_2", F.round("part_notes_1_2", 4))
               .withColumn("part_notes_5", F.round("part_notes_5", 4)))

m3_global = kpi_avis(m3_base.withColumn("tous", F.lit("toutes_livrees")), "tous")
m3_statut = kpi_avis(m3_base.withColumn("statut", F.when(F.col("is_late"), "en_retard").otherwise("a_l_heure")), "statut")
m3_tranche = kpi_avis(m3_base, "tranche_retard")

m3_global.show(truncate=False)
m3_statut.orderBy("statut").show(truncate=False)
m3_tranche.orderBy("tranche_retard").show(truncate=False)
print("Somme n_avis par tranche :", m3_tranche.agg(F.sum("n_avis")).first()[0], "(doit égaler la ligne « avec un avis » de la cellule 20)")

+--------------+------+------------+----------+--------------+------------+------------+
|tous          |n_avis|note_moyenne|ecart_type|part_notes_1_2|part_notes_5|marge95_note|
+--------------+------+------------+----------+--------------+------------+------------+
|toutes_livrees|95545 |4.156       |1.284     |0.128         |0.5923      |0.008       |
+--------------+------+------------+----------+--------------+------------+------------+

+---------+------+------------+----------+--------------+------------+------------+
|statut   |n_avis|note_moyenne|ecart_type|part_notes_1_2|part_notes_5|marge95_note|
+---------+------+------------+----------+--------------+------------+------------+
|a_l_heure|89168 |4.291       |1.148     |0.0925        |0.6228      |0.008       |
|en_retard|6377  |2.27        |1.571     |0.6243        |0.1653      |0.039       |
+---------+------+------------+----------+--------------+------------+------------+

+------------------------+------+------------+---

In [23]:
m3s = (m3_base
    .withColumn("region", region_map[F.col("customer_state")])
    .withColumn("nb_vendeurs", F.when(F.col("n_sellers") > 1, "plusieurs_vendeurs").otherwise("un_vendeur")))

def strate(df, cle):
    late = F.col("is_late")
    return (df.groupBy(cle).agg(
            F.count("*").alias("n"),
            F.sum(late.cast("int")).alias("n_retard"),
            F.round(F.avg(F.when(~late, F.col("review_score"))), 3).alias("note_a_l_heure"),
            F.round(F.avg(F.when(late, F.col("review_score"))), 3).alias("note_en_retard"))
        .withColumn("ecart", F.round(F.col("note_en_retard") - F.col("note_a_l_heure"), 3)))

print("Par région")
strate(m3s, "region").orderBy("region").show(truncate=False)
print("Par nombre de vendeurs")
strate(m3s, "nb_vendeurs").show(truncate=False)

# Moment de l'avis : un avis posté avant la livraison reflète-t-il l'expérience de livraison ?
oe = lire_silver("orders_enriched")
moment = (oe.filter("order_status = 'delivered' AND in_analysis_window AND has_review")
    .withColumn("avis_avant_livraison",
                F.to_date("review_creation_date") < F.to_date("order_delivered_customer_date")))
print("Population du contrôle :", moment.count(), "(attendu 95545)")
(moment.groupBy("is_late")
    .agg(F.count("*").alias("n"),
         F.round(F.avg(F.col("avis_avant_livraison").cast("int")), 4).alias("part_avis_avant_livraison"))
    .show())

Par région
+------------+-----+--------+--------------+--------------+------+
|region      |n    |n_retard|note_a_l_heure|note_en_retard|ecart |
+------------+-----+--------+--------------+--------------+------+
|Centro-Oeste|5580 |361     |4.263         |2.183         |-2.08 |
|Nordeste    |8945 |1121    |4.224         |2.181         |-2.043|
|Norte       |1774 |148     |4.203         |2.142         |-2.061|
|Sudeste     |65549|3948    |4.301         |2.288         |-2.013|
|Sul         |13697|799     |4.305         |2.372         |-1.933|
+------------+-----+--------+--------------+--------------+------+

Par nombre de vendeurs
+------------------+-----+--------+--------------+--------------+------+
|nb_vendeurs       |n    |n_retard|note_a_l_heure|note_en_retard|ecart |
+------------------+-----+--------+--------------+--------------+------+
|un_vendeur        |94287|6365    |4.311         |2.271         |-2.04 |
|plusieurs_vendeurs|1258 |12      |2.872         |1.833         |-1.03

In [24]:
ecrire_gold(m3_statut, "kpi_m3_statut")
ecrire_gold(m3_tranche, "kpi_m3_tranche")

s = spark.read.parquet(GOLD + "kpi_m3_statut")
t = spark.read.parquet(GOLD + "kpi_m3_tranche")
print("kpi_m3_statut  : lignes", s.count(), "(attendu 2), n_avis =", s.agg(F.sum("n_avis")).first()[0], "(attendu 95545)")
print("kpi_m3_tranche : lignes", t.count(), "(attendu 5), n_avis =", t.agg(F.sum("n_avis")).first()[0], "(attendu 95545)")

moy_pond = t.agg(F.sum(F.col("note_moyenne") * F.col("n_avis")) / F.sum("n_avis")).first()[0]
print(f"Note moyenne recalculée depuis les tranches : {moy_pond:.4f} (global mesuré : 4.156, écart attendu < 0.001)")

kpi_m3_statut  : lignes 2 (attendu 2), n_avis = 95545 (attendu 95545)
kpi_m3_tranche : lignes 5 (attendu 5), n_avis = 95545 (attendu 95545)
Note moyenne recalculée depuis les tranches : 4.1561 (global mesuré : 4.156, écart attendu < 0.001)


In [25]:
avis_ts = lire_silver("orders_enriched").select("order_id", "review_creation_date")

m3_t = (m3_base.join(avis_ts, "order_id", "left")
    .withColumn("avis_apres_livraison",
                F.to_date("review_creation_date") >= F.to_date("order_delivered_customer_date"))
    .withColumn("statut", F.when(F.col("is_late"), "en_retard").otherwise("a_l_heure")))

print("Lignes                                   :", m3_t.count(), "(attendu 95545)")
print("review_creation_date NULL                :", m3_t.filter("review_creation_date IS NULL").count(), "(attendu 0)")
n_apres = m3_t.filter("avis_apres_livraison").count()
print("Avis créés le jour de la livraison ou après :", n_apres, "(mesure)")
print("Contrôle croisé avec la cellule 22 (avant)  :", moment.filter("avis_avant_livraison").count(),
      "+", n_apres, "= 95545 ?")

m3_apres = m3_t.filter("avis_apres_livraison")

print("\nPar statut, avis postérieurs à la livraison")
kpi_avis(m3_apres, "statut").orderBy("statut").show(truncate=False)
print("Par tranche de retard, avis postérieurs à la livraison")
kpi_avis(m3_apres, "tranche_retard").orderBy("tranche_retard").show(truncate=False)

Lignes                                   : 95545 (attendu 95545)
review_creation_date NULL                : 0 (attendu 0)
Avis créés le jour de la livraison ou après : 90576 (mesure)
Contrôle croisé avec la cellule 22 (avant)  : 4969 + 90576 = 95545 ?

Par statut, avis postérieurs à la livraison
+---------+------+------------+----------+--------------+------------+------------+
|statut   |n_avis|note_moyenne|ecart_type|part_notes_1_2|part_notes_5|marge95_note|
+---------+------+------------+----------+--------------+------------+------------+
|a_l_heure|88947 |4.292       |1.147     |0.0923        |0.623       |0.008       |
|en_retard|1629  |3.526       |1.498     |0.2511        |0.3708      |0.073       |
+---------+------+------------+----------+--------------+------------+------------+

Par tranche de retard, avis postérieurs à la livraison
+------------------------+------+------------+----------+--------------+------------+------------+
|tranche_retard          |n_avis|note_moyenn

In [26]:
def avec_perimetre(df, nom):
    return df.withColumn("perimetre", F.lit(nom))

statut_all = avec_perimetre(kpi_avis(m3_t, "statut"), "tous_avis")
statut_apr = avec_perimetre(kpi_avis(m3_apres, "statut"), "avis_apres_livraison")
tranche_all = avec_perimetre(kpi_avis(m3_t, "tranche_retard"), "tous_avis")
tranche_apr = avec_perimetre(kpi_avis(m3_apres, "tranche_retard"), "avis_apres_livraison")

ecrire_gold(statut_all.unionByName(statut_apr), "kpi_m3_statut")
ecrire_gold(tranche_all.unionByName(tranche_apr), "kpi_m3_tranche")

s = spark.read.parquet(GOLD + "kpi_m3_statut")
t = spark.read.parquet(GOLD + "kpi_m3_tranche")
print("kpi_m3_statut  : lignes", s.count(), "(attendu 4)")
print("kpi_m3_tranche : lignes", t.count(), "(attendu 10)")
(s.groupBy("perimetre").agg(F.sum("n_avis").alias("n_avis")).show())
(t.groupBy("perimetre").agg(F.sum("n_avis").alias("n_avis")).show())
print("Attendu : tous_avis = 95545 ; avis_apres_livraison =", n_apres, "(des deux côtés)")

kpi_m3_statut  : lignes 4 (attendu 4)
kpi_m3_tranche : lignes 9 (attendu 10)
+--------------------+------+
|           perimetre|n_avis|
+--------------------+------+
|avis_apres_livraison| 90576|
|           tous_avis| 95545|
+--------------------+------+

+--------------------+------+
|           perimetre|n_avis|
+--------------------+------+
|avis_apres_livraison| 90576|
|           tous_avis| 95545|
+--------------------+------+

Attendu : tous_avis = 95545 ; avis_apres_livraison = 90576 (des deux côtés)


In [27]:
fo = spark.read.parquet(GOLD + "fact_orders")
cmd = fo.filter("is_sale")
print("Commandes is_sale :", cmd.count(), "(attendu 98176)")

clients = (cmd.groupBy("customer_unique_id")
    .agg(F.count("*").alias("n_commandes"),
         F.round(F.sum("items_total"), 2).alias("montant"),
         F.min("order_purchase_timestamp").alias("premiere_commande"),
         F.max("order_purchase_timestamp").alias("derniere_commande")))
clients.cache()

n_clients = clients.count()
n_recurrents = clients.filter("n_commandes >= 2").count()
print("Clients avec au moins une vente   :", n_clients, "(au plus 96067)")
print("Somme des commandes               :", clients.agg(F.sum("n_commandes")).first()[0], "(attendu 98176)")
print(f"Somme des montants                : {clients.agg(F.sum('montant')).first()[0]:,.2f} (attendu 13 491 302,13)")
print("Clients à 2 commandes ou plus     :", n_recurrents, "(au plus 2997)")
print(f"Taux de réachat (clients)         : {n_recurrents / n_clients:.4f}")
print(f"Part des commandes au-delà de la 1re : {(98176 - n_clients) / 98176:.4f}")
clients.groupBy("n_commandes").count().orderBy("n_commandes").show()

Commandes is_sale : 98176 (attendu 98176)
Clients avec au moins une vente   : 94961 (au plus 96067)
Somme des commandes               : 98176 (attendu 98176)
Somme des montants                : 13,491,302.13 (attendu 13 491 302,13)
Clients à 2 commandes ou plus     : 2886 (au plus 2997)
Taux de réachat (clients)         : 0.0304
Part des commandes au-delà de la 1re : 0.0327
+-----------+-----+
|n_commandes|count|
+-----------+-----+
|          1|92075|
|          2| 2650|
|          3|  188|
|          4|   29|
|          5|    9|
|          6|    5|
|          7|    3|
|          9|    1|
|         16|    1|
+-----------+-----+



In [28]:
w = Window.partitionBy("customer_unique_id").orderBy("order_purchase_timestamp", "order_id")

suite = (cmd.withColumn("achat_precedent", F.lag("order_purchase_timestamp").over(w))
    .filter("achat_precedent IS NOT NULL")
    .withColumn("jours_depuis_precedente",
                (F.unix_timestamp("order_purchase_timestamp") - F.unix_timestamp("achat_precedent")) / 86400))

n_suite = suite.count()
print("Commandes qui suivent une précédente :", n_suite, "(attendu", 98176 - n_clients, "= commandes - clients)")

(suite.agg(
    F.round(F.expr("percentile_approx(jours_depuis_precedente, 0.25)"), 2).alias("p25"),
    F.round(F.expr("percentile_approx(jours_depuis_precedente, 0.5)"), 2).alias("mediane"),
    F.round(F.expr("percentile_approx(jours_depuis_precedente, 0.75)"), 2).alias("p75"),
    F.round(F.max("jours_depuis_precedente"), 2).alias("max"),
    F.sum((F.col("jours_depuis_precedente") < 1).cast("int")).alias("moins_de_24h"))
    .show())

Commandes qui suivent une précédente : 3215 (attendu 3215 = commandes - clients)
+---+-------+------+------+------------+
|p25|mediane|   p75|   max|moins_de_24h|
+---+-------+------+------+------------+
|0.0|  29.01|120.66|608.98|         965|
+---+-------+------+------+------------+



In [29]:
clients2 = clients.withColumn(
    "ecart_premiere_derniere_j",
    (F.unix_timestamp("derniere_commande") - F.unix_timestamp("premiere_commande")) / 86400)

lignes = []
for libelle, s in [("tout réachat", 0.0), ("au moins 1 h", 1/24), ("au moins 1 jour", 1.0),
                   ("au moins 7 jours", 7.0), ("au moins 30 jours", 30.0)]:
    n = clients2.filter(f"n_commandes >= 2 AND ecart_premiere_derniere_j >= {s}").count()
    lignes.append((libelle, s, n, round(n / n_clients, 4)))

spark.createDataFrame(lignes, ["seuil", "jours", "n_clients_reacheteurs", "taux_reachat"]).show(truncate=False)
print("La ligne « tout réachat » doit donner 2886 :", lignes[0][2])

+-----------------+--------------------+---------------------+------------+
|seuil            |jours               |n_clients_reacheteurs|taux_reachat|
+-----------------+--------------------+---------------------+------------+
|tout réachat     |0.0                 |2886                 |0.0304      |
|au moins 1 h     |0.041666666666666664|2095                 |0.0221      |
|au moins 1 jour  |1.0                 |2038                 |0.0215      |
|au moins 7 jours |7.0                 |1877                 |0.0198      |
|au moins 30 jours|30.0                |1490                 |0.0157      |
+-----------------+--------------------+---------------------+------------+

La ligne « tout réachat » doit donner 2886 : 2886


In [30]:
H = 90
FIN = fo.agg(F.max("order_purchase_timestamp")).first()[0]
print("Fin du dataset :", FIN)

cohorte = clients.filter(
    F.unix_timestamp("premiere_commande") <= F.unix_timestamp(F.lit(FIN)) - H * 86400)

apres = (cmd.select("customer_unique_id", F.col("order_purchase_timestamp").alias("ts"))
    .join(clients.select("customer_unique_id", "premiere_commande"), "customer_unique_id")
    .withColumn("delta_j", (F.unix_timestamp("ts") - F.unix_timestamp("premiere_commande")) / 86400))

revenus = (apres.filter(f"delta_j >= 1 AND delta_j <= {H}")
    .select("customer_unique_id").distinct().withColumn("rev", F.lit(1)))

c = cohorte.join(revenus, "customer_unique_id", "left").fillna({"rev": 0})

print("Clients de la cohorte :", c.count(), "(mesure ; inférieur à", n_clients, ")")
print("Taux de réachat à 90 jours (réachat à 1 jour ou plus) :", round(c.agg(F.avg("rev")).first()[0], 4))

(c.groupBy(F.date_format("premiere_commande", "yyyy-MM").alias("mois_premiere_commande"))
  .agg(F.count("*").alias("n_clients"), F.round(F.avg("rev"), 4).alias("taux_reachat_90j"))
  .orderBy("mois_premiere_commande").show(30))

Fin du dataset : 2018-10-17 17:30:18
Clients de la cohorte : 85886 (mesure ; inférieur à 94961 )
Taux de réachat à 90 jours (réachat à 1 jour ou plus) : 0.0124
+----------------------+---------+----------------+
|mois_premiere_commande|n_clients|taux_reachat_90j|
+----------------------+---------+----------------+
|               2016-09|        2|             0.0|
|               2016-10|      287|             0.0|
|               2016-12|        1|             1.0|
|               2017-01|      750|          0.0093|
|               2017-02|     1678|          0.0095|
|               2017-03|     2571|          0.0132|
|               2017-04|     2325|          0.0129|
|               2017-05|     3540|          0.0158|
|               2017-06|     3102|          0.0139|
|               2017-07|     3822|          0.0128|
|               2017-08|     4130|          0.0143|
|               2017-09|     4074|          0.0182|
|               2017-10|     4393|          0.0121|
|       

In [31]:
SEUIL_SESSION_H = 24
w = Window.partitionBy("customer_unique_id").orderBy("order_purchase_timestamp", "order_id")

cmd_s = (cmd.select("customer_unique_id", "order_id", "order_purchase_timestamp", "items_total")
    .withColumn("prev_ts", F.lag("order_purchase_timestamp").over(w))
    .withColumn("nouvelle_session",
        F.when(F.col("prev_ts").isNull(), 1)
         .when(F.unix_timestamp("order_purchase_timestamp") - F.unix_timestamp("prev_ts")
               >= SEUIL_SESSION_H * 3600, 1)
         .otherwise(0))
    .withColumn("session_id",
                F.sum("nouvelle_session").over(w.rowsBetween(Window.unboundedPreceding, Window.currentRow))))

par_client = (cmd_s.groupBy("customer_unique_id")
    .agg(F.max("session_id").alias("n_sessions"), F.count("*").alias("n_commandes")))

print("Clients                        :", par_client.count(), "(attendu 94961)")
print("Somme des commandes            :", par_client.agg(F.sum("n_commandes")).first()[0], "(attendu 98176)")
print("Clients à 2 sessions ou plus   :", par_client.filter("n_sessions >= 2").count(), "(au plus 2038)")
print("Clients avec commandes fusionnées (n_commandes > n_sessions) :",
      par_client.filter("n_commandes > n_sessions").count(), "(mesure)")
par_client.groupBy("n_sessions").count().orderBy("n_sessions").show()

Clients                        : 94961 (attendu 94961)
Somme des commandes            : 98176 (attendu 98176)
Clients à 2 sessions ou plus   : 2036 (au plus 2038)
Clients avec commandes fusionnées (n_commandes > n_sessions) : 924 (mesure)
+----------+-----+
|n_sessions|count|
+----------+-----+
|         1|92925|
|         2| 1884|
|         3|  122|
|         4|   16|
|         5|    7|
|         6|    5|
|         7|    1|
|        16|    1|
+----------+-----+



In [32]:
REF = F.date_add(F.to_date(F.lit(FIN)), 1)      # lendemain du dernier achat du dataset

rfm = (cmd_s.groupBy("customer_unique_id")
    .agg(F.max("session_id").alias("frequence"),
         F.round(F.sum("items_total"), 2).alias("montant"),
         F.max("order_purchase_timestamp").alias("derniere_commande"))
    .withColumn("recence_j", F.datediff(REF, F.to_date("derniere_commande"))))
rfm.cache()

print("Clients                 :", rfm.count(), "(attendu 94961)")
print(f"Somme des montants      : {rfm.agg(F.sum('montant')).first()[0]:,.2f} (attendu 13 491 302,13)")
print("Montant <= 0            :", rfm.filter("montant <= 0").count(), "(attendu 0)")
print("Récence min / max       :", rfm.agg(F.min('recence_j'), F.max('recence_j')).first(), "(min attendu 1 ; max au plus 774)")

rfm.groupBy("frequence").count().orderBy("frequence").show()
rfm.select(
    F.expr("percentile_approx(recence_j, array(0.2, 0.4, 0.6, 0.8))").alias("quintiles_recence"),
    F.expr("percentile_approx(montant, array(0.2, 0.4, 0.6, 0.8))").alias("quintiles_montant")
).show(truncate=False)

Clients                 : 94961 (attendu 94961)
Somme des montants      : 13,491,302.13 (attendu 13 491 302,13)
Montant <= 0            : 0 (attendu 0)
Récence min / max       : Row(min(recence_j)=45, max(recence_j)=774) (min attendu 1 ; max au plus 774)
+---------+-----+
|frequence|count|
+---------+-----+
|        1|92925|
|        2| 1884|
|        3|  122|
|        4|   16|
|        5|    7|
|        6|    5|
|        7|    1|
|       16|    1|
+---------+-----+

+--------------------+--------------------------+
|quintiles_recence   |quintiles_montant         |
+--------------------+--------------------------+
|[143, 227, 318, 433]|[39.9, 69.9, 109.9, 179.9]|
+--------------------+--------------------------+



In [33]:
FIN_VENTES = cmd.agg(F.max("order_purchase_timestamp")).first()[0]
print("Dernière vente (is_sale)                 :", FIN_VENTES)
print("Dernière commande tous statuts           :", FIN)
print("Commandes hors vente après la dernière vente :",
      fo.filter((~F.col("is_sale")) & (F.col("order_purchase_timestamp") > F.lit(FIN_VENTES))).count(), "(mesure)")

# --- Récence recalculée ---
REF = F.date_add(F.to_date(F.lit(FIN_VENTES)), 1)
rfm = (cmd_s.groupBy("customer_unique_id")
    .agg(F.max("session_id").alias("frequence"),
         F.round(F.sum("items_total"), 2).alias("montant"),
         F.max("order_purchase_timestamp").alias("derniere_commande"))
    .withColumn("recence_j", F.datediff(REF, F.to_date("derniere_commande"))))
rfm.cache()

print("Récence min / max :", rfm.agg(F.min("recence_j"), F.max("recence_j")).first(),
      "(attendu 1 / 730 si la dernière vente est le 2018-09-03 : 45 - 44 et 774 - 44)")
rfm.select(F.expr("percentile_approx(recence_j, array(0.2, 0.4, 0.6, 0.8))").alias("quintiles_recence")
          ).show(truncate=False)
print("Attendu : [99, 183, 274, 389] (anciens quintiles - 44)")

# --- Cohorte à 90 jours recalculée ---
cohorte = clients.filter(F.unix_timestamp("premiere_commande") <= F.unix_timestamp(F.lit(FIN_VENTES)) - H * 86400)
c = cohorte.join(revenus, "customer_unique_id", "left").fillna({"rev": 0})
print("Clients de la cohorte :", c.count(), "(avant correction : 85886 ; doit être inférieur)")
print("Taux de réachat à 90 jours :", round(c.agg(F.avg("rev")).first()[0], 4), "(avant correction : 0.0124)")
(c.groupBy(F.date_format("premiere_commande", "yyyy-MM").alias("mois_premiere_commande"))
  .agg(F.count("*").alias("n_clients"), F.round(F.avg("rev"), 4).alias("taux_reachat_90j"))
  .orderBy("mois_premiere_commande").show(30))

Dernière vente (is_sale)                 : 2018-09-03 09:06:57
Dernière commande tous statuts           : 2018-10-17 17:30:18
Commandes hors vente après la dernière vente : 19 (mesure)
Récence min / max : Row(min(recence_j)=1, max(recence_j)=730) (attendu 1 / 730 si la dernière vente est le 2018-09-03 : 45 - 44 et 774 - 44)
+-------------------+
|quintiles_recence  |
+-------------------+
|[99, 183, 274, 389]|
+-------------------+

Attendu : [99, 183, 274, 389] (anciens quintiles - 44)
Clients de la cohorte : 77544 (avant correction : 85886 ; doit être inférieur)
Taux de réachat à 90 jours : 0.0128 (avant correction : 0.0124)
+----------------------+---------+----------------+
|mois_premiere_commande|n_clients|taux_reachat_90j|
+----------------------+---------+----------------+
|               2016-09|        2|             0.0|
|               2016-10|      287|             0.0|
|               2016-12|        1|             1.0|
|               2017-01|      750|          0.0093|
|

In [34]:
q_r = rfm.select(F.expr("percentile_approx(recence_j, array(0.2, 0.4, 0.6, 0.8))")).first()[0]
q_m = rfm.select(F.expr("percentile_approx(montant, array(0.2, 0.4, 0.6, 0.8))")).first()[0]
print("Seuils de récence :", q_r, "| seuils de montant :", q_m)

rec, mon = F.col("recence_j"), F.col("montant")
score_r = (F.when(rec <= q_r[0], 5).when(rec <= q_r[1], 4).when(rec <= q_r[2], 3)
            .when(rec <= q_r[3], 2).otherwise(1))
score_m = (F.when(mon <= q_m[0], 1).when(mon <= q_m[1], 2).when(mon <= q_m[2], 3)
            .when(mon <= q_m[3], 4).otherwise(5))
score_f = F.when(F.col("frequence") >= 3, 3).otherwise(F.col("frequence")).cast("int")

rfm_s = (rfm.withColumn("score_r", score_r).withColumn("score_f", score_f).withColumn("score_m", score_m)
    .withColumn("segment",
        F.when((F.col("score_f") >= 2) & (F.col("score_r") >= 4), "fideles_actifs")
         .when(F.col("score_f") >= 2, "fideles_a_reactiver")
         .when((F.col("score_r") >= 4) & (F.col("score_m") >= 4), "nouveaux_a_forte_valeur")
         .when(F.col("score_r") >= 4, "nouveaux")
         .when(F.col("score_m") >= 4, "occasionnels_a_forte_valeur_a_reactiver")
         .otherwise("occasionnels_inactifs")))
rfm_s.cache()

n_tot = rfm_s.count()
ca_tot = rfm_s.agg(F.sum("montant")).first()[0]
print("Clients scorés        :", n_tot, "(attendu 94961)")
print("Segment NULL          :", rfm_s.filter("segment IS NULL").count(), "(attendu 0)")
print("Scores hors plage     :", rfm_s.filter("score_r NOT BETWEEN 1 AND 5 OR score_m NOT BETWEEN 1 AND 5 "
                                              "OR score_f NOT BETWEEN 1 AND 3").count(), "(attendu 0)")

for s in ["score_r", "score_m", "score_f"]:
    rfm_s.groupBy(s).count().orderBy(s).show()

seg = (rfm_s.groupBy("segment").agg(
        F.count("*").alias("n_clients"),
        F.round(F.sum("montant"), 2).alias("ca"),
        F.round(F.avg("montant"), 2).alias("montant_moyen"),
        F.round(F.avg("recence_j"), 1).alias("recence_moyenne"),
        F.round(F.avg("frequence"), 3).alias("frequence_moyenne"))
    .withColumn("part_clients", F.round(F.col("n_clients") / n_tot, 4))
    .withColumn("part_ca", F.round(F.col("ca") / ca_tot, 4))
    .orderBy(F.desc("ca")))
seg.show(truncate=False)
print("Somme n_clients :", seg.agg(F.sum("n_clients")).first()[0], "(attendu 94961)")
print(f"Somme CA        : {seg.agg(F.sum('ca')).first()[0]:,.2f} (attendu 13 491 302,13)")

Seuils de récence : [99, 183, 274, 389] | seuils de montant : [39.9, 69.9, 109.9, 179.9]
Clients scorés        : 94961 (attendu 94961)
Segment NULL          : 0 (attendu 0)
Scores hors plage     : 0 (attendu 0)
+-------+-----+
|score_r|count|
+-------+-----+
|      1|18952|
|      2|18925|
|      3|19005|
|      4|18952|
|      5|19127|
+-------+-----+

+-------+-----+
|score_m|count|
+-------+-----+
|      1|19639|
|      2|19730|
|      3|17720|
|      4|18969|
|      5|18903|
+-------+-----+

+-------+-----+
|score_f|count|
+-------+-----+
|      1|92925|
|      2| 1884|
|      3|  152|
+-------+-----+

+---------------------------------------+---------+----------+-------------+---------------+-----------------+------------+-------+
|segment                                |n_clients|ca        |montant_moyen|recence_moyenne|frequence_moyenne|part_clients|part_ca|
+---------------------------------------+---------+----------+-------------+---------------+-----------------+------------

In [35]:
rfm_out = rfm_s.select("customer_unique_id", "frequence", "montant", "derniere_commande",
                       "recence_j", "score_r", "score_f", "score_m", "segment")
reachat_seuil = spark.createDataFrame(lignes, ["seuil", "jours", "n_clients_reacheteurs", "taux_reachat"])
reachat_cohorte = (c.groupBy(F.date_format("premiere_commande", "yyyy-MM").alias("mois_premiere_commande"))
    .agg(F.count("*").alias("n_clients"), F.round(F.avg("rev"), 4).alias("taux_reachat_90j"))
    .withColumn("horizon_j", F.lit(H))
    .withColumn("fin_reference", F.lit(str(FIN_VENTES))))

ecrire_gold(rfm_out, "kpi_m4_rfm")
ecrire_gold(seg, "kpi_m4_segment")
ecrire_gold(reachat_seuil, "kpi_m4_reachat_seuil")
ecrire_gold(reachat_cohorte, "kpi_m4_reachat_cohorte")

r = spark.read.parquet(GOLD + "kpi_m4_rfm")
sg = spark.read.parquet(GOLD + "kpi_m4_segment")
rs = spark.read.parquet(GOLD + "kpi_m4_reachat_seuil")
rc = spark.read.parquet(GOLD + "kpi_m4_reachat_cohorte")

print("kpi_m4_rfm              : lignes", r.count(), "/ clients distincts",
      r.select("customer_unique_id").distinct().count(), "(attendu 94961 / 94961)")
print(f"  somme des montants    : {r.agg(F.sum('montant')).first()[0]:,.2f} (attendu 13 491 302,13)")
print("kpi_m4_segment          : lignes", sg.count(), "(attendu 6), clients =",
      sg.agg(F.sum("n_clients")).first()[0], "(attendu 94961)")
print("kpi_m4_reachat_seuil    : lignes", rs.count(), "(attendu 5)")
print("kpi_m4_reachat_cohorte  : lignes", rc.count(), "(attendu 21), clients =",
      rc.agg(F.sum("n_clients")).first()[0], "(attendu 77544)")
pond = rc.agg(F.sum(F.col("n_clients") * F.col("taux_reachat_90j")) / F.sum("n_clients")).first()[0]
print(f"  taux pondéré relu     : {pond:.4f} (attendu environ 0.0128)")

kpi_m4_rfm              : lignes 94961 / clients distincts 94961 (attendu 94961 / 94961)
  somme des montants    : 13,491,302.13 (attendu 13 491 302,13)
kpi_m4_segment          : lignes 6 (attendu 6), clients = 94961 (attendu 94961)
kpi_m4_reachat_seuil    : lignes 5 (attendu 5)
kpi_m4_reachat_cohorte  : lignes 21 (attendu 21), clients = 77544 (attendu 77544)
  taux pondéré relu     : 0.0128 (attendu environ 0.0128)


In [36]:
fo = spark.read.parquet(GOLD + "fact_orders")
n_cmd = fo.count()
print("Commandes (Silver) :", n_cmd, "(attendu 99410)")

statuts = (fo.groupBy("order_status")
    .agg(F.count("*").alias("n"), F.round(F.sum("items_total"), 2).alias("items_total"))
    .withColumn("part", F.round(F.col("n") / n_cmd, 4)).orderBy(F.desc("n")))
statuts.show(truncate=False)

obs = {r["order_status"]: r["n"] for r in statuts.collect()}
att = {"delivered": 96455, "shipped": 1106, "canceled": 625, "unavailable": 609,
       "invoiced": 312, "processing": 301, "approved": 2, "created": 0}
for s, a in att.items():
    print(f"  {s:12} observé {obs.get(s, 0):>6} | attendu {a:>6}")

na = fo.filter("order_status IN ('canceled', 'unavailable')")
print("\ncanceled + unavailable          :", na.count(), "(attendu 1234)")
print(f"Taux de commandes non abouties  : {na.count() / n_cmd:.4f}")
print(f"CA associé (items_total)        : {na.agg(F.sum('items_total')).first()[0]:,.2f} (attendu 97 242,96)")
print("  canceled avec produits        :", fo.filter("order_status = 'canceled' AND has_items").count(), "(attendu 461)")
print("  unavailable avec produits     :", fo.filter("order_status = 'unavailable' AND has_items").count(), "(attendu 6)")

Commandes (Silver) : 99410 (attendu 99410)
+------------+-----+------------+------+
|order_status|n    |items_total |part  |
+------------+-----+------------+------+
|delivered   |96455|1.32183995E7|0.9703|
|shipped     |1106 |150727.44   |0.0111|
|canceled    |625  |95235.27    |0.0063|
|unavailable |609  |2007.69     |0.0061|
|invoiced    |312  |61526.37    |0.0031|
|processing  |301  |60439.22    |0.003 |
|approved    |2    |209.6       |0.0   |
+------------+-----+------------+------+

  delivered    observé  96455 | attendu  96455
  shipped      observé   1106 | attendu   1106
  canceled     observé    625 | attendu    625
  unavailable  observé    609 | attendu    609
  invoiced     observé    312 | attendu    312
  processing   observé    301 | attendu    301
  approved     observé      2 | attendu      2
  created      observé      0 | attendu      0

canceled + unavailable          : 1234 (attendu 1234)
Taux de commandes non abouties  : 0.0124
CA associé (items_total)        :

In [37]:
def wilson(df, k, n):
    p = F.col(k) / F.col(n)
    d = 1 + (Z ** 2) / F.col(n)
    centre = (p + (Z ** 2) / (2 * F.col(n))) / d
    marge = Z * F.sqrt(p * (1 - p) / F.col(n) + (Z ** 2) / (4 * F.col(n) * F.col(n))) / d
    return (df.withColumn("taux", F.round(p, 4))
              .withColumn("ic95_bas", F.round(centre - marge, 4))
              .withColumn("ic95_haut", F.round(centre + marge, 4)))

mensuel_m5 = (fo.filter("in_analysis_window")
    .groupBy(F.date_format("order_purchase_timestamp", "yyyy-MM").alias("year_month"))
    .agg(F.count("*").alias("n_commandes"),
         F.sum((F.col("order_status") == "canceled").cast("int")).alias("n_canceled"),
         F.sum((F.col("order_status") == "unavailable").cast("int")).alias("n_unavailable"))
    .withColumn("n_non_abouties", F.col("n_canceled") + F.col("n_unavailable")))
mensuel_m5 = wilson(mensuel_m5, "n_non_abouties", "n_commandes").orderBy("year_month")

mensuel_m5.show(25, truncate=False)
print("Mois                 :", mensuel_m5.count(), "(attendu 20)")
print("Somme des commandes  :", mensuel_m5.agg(F.sum("n_commandes")).first()[0], "(attendu 99064)")

+----------+-----------+----------+-------------+--------------+------+--------+---------+
|year_month|n_commandes|n_canceled|n_unavailable|n_non_abouties|taux  |ic95_bas|ic95_haut|
+----------+-----------+----------+-------------+--------------+------+--------+---------+
|2017-01   |798        |3         |10           |13            |0.0163|0.0095  |0.0277   |
|2017-02   |1768       |17        |45           |62            |0.0351|0.0275  |0.0447   |
|2017-03   |2682       |33        |32           |65            |0.0242|0.0191  |0.0308   |
|2017-04   |2404       |18        |9            |27            |0.0112|0.0077  |0.0163   |
|2017-05   |3699       |29        |31           |60            |0.0162|0.0126  |0.0208   |
|2017-06   |3245       |16        |24           |40            |0.0123|0.0091  |0.0167   |
|2017-07   |4026       |28        |52           |80            |0.0199|0.016   |0.0247   |
|2017-08   |4331       |27        |32           |59            |0.0136|0.0106  |0.0175   |

In [38]:
q = lire_silver("quarantine_orders")
print("Quarantaine par motif (attendu 23 et 8) :")
q.groupBy("quarantine_reason").count().show(truncate=False)

print("canceled mais livrée (Silver)                     :", fo.filter("is_canceled_but_delivered").count(), "(attendu 6)")
print("transporteur avant approbation (Silver)           :", fo.filter("flag_carrier_before_approval").count(), "(au plus 1359)")
print("livraison avant transporteur (Silver)             :", fo.filter("flag_delivery_before_carrier").count(), "(au plus 23)")
print("sans produit, statut hors canceled/unavailable    :",
      fo.filter("NOT has_items AND order_status NOT IN ('canceled', 'unavailable')").count(), "(attendu 0)")
print("shipped sans livraison de plus de 60 j (is_shipped_stale) :", fo.filter("is_shipped_stale").count(), "(attendu 1098)")

# Statuts non finaux : ancienneté par rapport à la fin du dataset (comme R7)
FIN_DS = fo.agg(F.max("order_purchase_timestamp")).first()[0]
print("\nStatuts non finaux, ancienneté au", FIN_DS, "(mesure ; shipped > 60 j doit donner 1098)")
(fo.filter("order_status IN ('created', 'approved', 'invoiced', 'processing', 'shipped')")
   .withColumn("age_j", F.datediff(F.lit(FIN_DS), F.col("order_purchase_timestamp")))
   .groupBy("order_status")
   .agg(F.count("*").alias("n"),
        F.sum((F.col("age_j") > 60).cast("int")).alias("n_plus_60j"),
        F.min("order_purchase_timestamp").alias("achat_min"),
        F.max("order_purchase_timestamp").alias("achat_max"))
   .orderBy(F.desc("n")).show(truncate=False))

Quarantaine par motif (attendu 23 et 8) :
+----------------------------+-----+
|quarantine_reason           |count|
+----------------------------+-----+
|no_items_inconsistent_status|8    |
|delivered_missing_date      |23   |
+----------------------------+-----+

canceled mais livrée (Silver)                     : 6 (attendu 6)
transporteur avant approbation (Silver)           : 1359 (au plus 1359)
livraison avant transporteur (Silver)             : 23 (au plus 23)
sans produit, statut hors canceled/unavailable    : 0 (attendu 0)
shipped sans livraison de plus de 60 j (is_shipped_stale) : 1098 (attendu 1098)

Statuts non finaux, ancienneté au 2018-10-17 17:30:18 (mesure ; shipped > 60 j doit donner 1098)
+------------+----+----------+-------------------+-------------------+
|order_status|n   |n_plus_60j|achat_min          |achat_max          |
+------------+----+----------+-------------------+-------------------+
|shipped     |1106|1098      |2016-09-04 21:15:19|2018-09-03 09:06:57|
|

In [39]:
NON_FINAUX = "('created', 'approved', 'invoiced', 'processing', 'shipped')"

nf = (fo.filter("in_analysis_window")
    .groupBy(F.date_format("order_purchase_timestamp", "yyyy-MM").alias("year_month"))
    .agg(F.sum(F.col("order_status").isin("created", "approved", "invoiced", "processing", "shipped")
               .cast("int")).alias("n_non_finaux")))

mensuel_out = mensuel_m5.join(nf, "year_month").orderBy("year_month")
(mensuel_out.select("year_month", "n_commandes", "n_non_finaux",
                    F.round(F.col("n_non_finaux") / F.col("n_commandes"), 4).alias("part_non_finaux"))
            .show(25, truncate=False))

ca_nf = fo.filter(f"is_sale AND order_status IN {NON_FINAUX}").agg(F.sum("items_total")).first()[0]
print(f"CA des ventes porté par des commandes en statut non final : {ca_nf:,.2f} (attendu 272 902,63)")
print(f"  part du CA des ventes : {ca_nf / 13491302.13:.4f} (calculé : environ 0.0202)")

+----------+-----------+------------+---------------+
|year_month|n_commandes|n_non_finaux|part_non_finaux|
+----------+-----------+------------+---------------+
|2017-01   |798        |37          |0.0464         |
|2017-02   |1768       |65          |0.0368         |
|2017-03   |2682       |71          |0.0265         |
|2017-04   |2404       |74          |0.0308         |
|2017-05   |3699       |94          |0.0254         |
|2017-06   |3245       |70          |0.0216         |
|2017-07   |4026       |74          |0.0184         |
|2017-08   |4331       |79          |0.0182         |
|2017-09   |4284       |77          |0.018          |
|2017-10   |4631       |69          |0.0149         |
|2017-11   |7541       |132         |0.0175         |
|2017-12   |5671       |105         |0.0185         |
|2018-01   |7269       |118         |0.0162         |
|2018-02   |6727       |69          |0.0103         |
|2018-03   |7211       |165         |0.0229         |
|2018-04   |6939       |121 

In [40]:
non_finaux = (fo.filter(f"order_status IN {NON_FINAUX}")
    .withColumn("age_j", F.datediff(F.lit(FIN_DS), F.col("order_purchase_timestamp"))))
def vieux(statut):
    return non_finaux.filter(f"order_status = '{statut}' AND age_j > 60").count()

anomalies = spark.createDataFrame([
    ("quarantaine", "delivered_missing_date", q.filter("quarantine_reason = 'delivered_missing_date'").count()),
    ("quarantaine", "no_items_inconsistent_status", q.filter("quarantine_reason = 'no_items_inconsistent_status'").count()),
    ("indicateur", "canceled_but_delivered", fo.filter("is_canceled_but_delivered").count()),
    ("indicateur", "carrier_before_approval", fo.filter("flag_carrier_before_approval").count()),
    ("indicateur", "delivery_before_carrier", fo.filter("flag_delivery_before_carrier").count()),
    ("indicateur", "shipped_stale_60j", fo.filter("is_shipped_stale").count()),
    ("constat", "invoiced_plus_60j", vieux("invoiced")),
    ("constat", "processing_plus_60j", vieux("processing")),
    ("constat", "approved_plus_60j", vieux("approved")),
], ["famille", "anomalie", "n_commandes"])
anomalies.show(truncate=False)
print("Attendu : 23, 8, 6, 1359, 23, 1098, 312, 301, 2")

ecrire_gold(statuts, "kpi_m5_statut")
ecrire_gold(mensuel_out, "kpi_m5_mensuel")
ecrire_gold(anomalies, "kpi_m5_anomalies")

a = spark.read.parquet(GOLD + "kpi_m5_statut")
b = spark.read.parquet(GOLD + "kpi_m5_mensuel")
c5 = spark.read.parquet(GOLD + "kpi_m5_anomalies")
print("kpi_m5_statut    : lignes", a.count(), "(attendu 7), n =", a.agg(F.sum("n")).first()[0], "(attendu 99410)")
print(f"  items_total    : {a.agg(F.sum('items_total')).first()[0]:,.2f} (attendu 13 588 545,09)")
print("kpi_m5_mensuel   : lignes", b.count(), "(attendu 20), n =", b.agg(F.sum("n_commandes")).first()[0], "(attendu 99064)")
print("kpi_m5_anomalies : lignes", c5.count(), "(attendu 9)")

+-----------+----------------------------+-----------+
|famille    |anomalie                    |n_commandes|
+-----------+----------------------------+-----------+
|quarantaine|delivered_missing_date      |23         |
|quarantaine|no_items_inconsistent_status|8          |
|indicateur |canceled_but_delivered      |6          |
|indicateur |carrier_before_approval     |1359       |
|indicateur |delivery_before_carrier     |23         |
|indicateur |shipped_stale_60j           |1098       |
|constat    |invoiced_plus_60j           |312        |
|constat    |processing_plus_60j         |301        |
|constat    |approved_plus_60j           |2          |
+-----------+----------------------------+-----------+

Attendu : 23, 8, 6, 1359, 23, 1098, 312, 301, 2


kpi_m5_statut    : lignes 7 (attendu 7), n = 99410 (attendu 99410)
  items_total    : 13,588,545.09 (attendu 13 588 545,09)
kpi_m5_mensuel   : lignes 20 (attendu 20), n = 99064 (attendu 99064)
kpi_m5_anomalies : lignes 9 (attendu 9)


In [41]:
pa = lire_silver("order_payments")
print("Paiements (Silver)                  :", pa.count(), "(attendu 103855)")
print("Somme de n_payments (fact_orders)   :", fo.agg(F.sum("n_payments")).first()[0], "(attendu 103855)")
print("Commandes sans paiement             :", fo.filter("NOT has_payment").count(), "(attendu 1)")
print(f"Somme de payment_value (Silver)     : {pa.agg(F.sum('payment_value')).first()[0]:,.2f}")
print(f"Somme de payment_total (fact_orders): {fo.agg(F.sum('payment_total')).first()[0]:,.2f} (égal à la ligne précédente, à quelques centimes près)")
print("Lignes anormales (is_anomalous_payment) :", pa.filter("is_anomalous_payment").count(), "(au plus 11)")
pa.groupBy("payment_type").count().orderBy(F.desc("count")).show()

cmp = (fo.filter("n_items > 0 AND has_payment")
         .withColumn("ecart", F.round(F.col("payment_total") - F.col("items_total") - F.col("freight_total"), 2)))
ecarts = cmp.filter(F.abs(F.col("ecart")) > 0.01)
print("Commandes comparables               :", cmp.count(), "(attendu 98642)")
print("Écart de plus d'un centime          :", ecarts.count(), "(attendu 303)")

print("\nLes 303 commandes, par type de paiement")
(ecarts.groupBy("payment_types")
    .agg(F.count("*").alias("n"),
         F.sum((F.col("ecart") > 0).cast("int")).alias("n_positifs"),
         F.sum((F.col("ecart") < 0).cast("int")).alias("n_negatifs"),
         F.round(F.avg("ecart"), 2).alias("ecart_moyen"),
         F.round(F.min("ecart"), 2).alias("ecart_min"),
         F.round(F.max("ecart"), 2).alias("ecart_max"))
    .orderBy(F.desc("n")).show(truncate=False))

print("Taux d'écart dans chaque type de paiement (toutes les commandes comparables)")
(cmp.groupBy("payment_types")
    .agg(F.count("*").alias("n_total"),
         F.sum((F.abs(F.col("ecart")) > 0.01).cast("int")).alias("n_ecart"))
    .withColumn("taux_ecart", F.round(F.col("n_ecart") / F.col("n_total"), 4))
    .orderBy(F.desc("n_total")).show(truncate=False))

Paiements (Silver)                  : 103855 (attendu 103855)
Somme de n_payments (fact_orders)   : 103855 (attendu 103855)
Commandes sans paiement             : 1 (attendu 1)
Somme de payment_value (Silver)     : 16,004,429.56
Somme de payment_total (fact_orders): 16,004,429.56 (égal à la ligne précédente, à quelques centimes près)
Lignes anormales (is_anomalous_payment) : 11 (au plus 11)
+------------+-----+
|payment_type|count|
+------------+-----+
| credit_card|76783|
|      boleto|19766|
|     voucher| 5775|
|  debit_card| 1528|
|     unknown|    3|
+------------+-----+

Commandes comparables               : 98642 (attendu 98642)
Écart de plus d'un centime          : 303 (attendu 303)

Les 303 commandes, par type de paiement
+-------------------+---+----------+----------+-----------+---------+---------+
|payment_types      |n  |n_positifs|n_negatifs|ecart_moyen|ecart_min|ecart_max|
+-------------------+---+----------+----------+-----------+---------+---------+
|credit_card        

In [42]:
sale_ids = fo.filter("is_sale").select("order_id")
pa_sale = pa.join(sale_ids, "order_id", "left_semi")
tot_n = pa_sale.count()
tot_v = pa_sale.agg(F.sum("payment_value")).first()[0]
print("Paiements des commandes is_sale :", tot_n, "| valeur totale :", f"{tot_v:,.2f}", "(mesures)")

par_type = (pa_sale.groupBy("payment_type")
    .agg(F.count("*").alias("n_paiements"),
         F.countDistinct("order_id").alias("n_commandes"),
         F.round(F.sum("payment_value"), 2).alias("valeur"),
         F.round(F.avg("payment_installments"), 2).alias("echeances_moyennes"))
    .withColumn("part_paiements", F.round(F.col("n_paiements") / tot_n, 4))
    .withColumn("part_valeur", F.round(F.col("valeur") / tot_v, 4))
    .orderBy(F.desc("valeur")))
par_type.show(truncate=False)
print("Somme des paiements :", par_type.agg(F.sum("n_paiements")).first()[0], "(doit égaler la ligne « Paiements des commandes is_sale »)")

# Niveau commande : panier selon la combinaison de moyens de paiement
pay_cmd = fo.filter("is_sale AND has_payment")
print("\nCommandes is_sale avec paiement :", pay_cmd.count(), "(au plus 98176)")
par_combo = (pay_cmd.groupBy("payment_types")
    .agg(F.count("*").alias("n_commandes"),
         F.round(F.avg("items_total"), 2).alias("panier_moyen"),
         F.round(F.expr("percentile_approx(items_total, 0.5)"), 2).alias("panier_median"),
         F.round(F.avg("payment_total"), 2).alias("paiement_moyen"))
    .orderBy(F.desc("n_commandes")))
par_combo.show(truncate=False)

# Échéances : commandes payées uniquement par carte
cb = fo.filter("is_sale AND payment_types = 'credit_card'")
tr = (F.when(F.col("max_installments") <= 1, "1_une_echeance")
       .when(F.col("max_installments") <= 3, "2_2_a_3")
       .when(F.col("max_installments") <= 6, "3_4_a_6")
       .when(F.col("max_installments") <= 10, "4_7_a_10")
       .otherwise("5_11_et_plus"))
par_echeances = (cb.withColumn("tranche_echeances", tr).groupBy("tranche_echeances")
    .agg(F.count("*").alias("n_commandes"),
         F.round(F.avg("items_total"), 2).alias("panier_moyen"),
         F.round(F.expr("percentile_approx(items_total, 0.5)"), 2).alias("panier_median"))
    .orderBy("tranche_echeances"))
print("Commandes payées uniquement par carte :", cb.count(), "(mesure)")
par_echeances.show(truncate=False)
print("Somme des tranches :", par_echeances.agg(F.sum("n_commandes")).first()[0], "(doit égaler la ligne précédente)")

Paiements des commandes is_sale : 102542 | valeur totale : 15,734,694.45 (mesures)
+------------+-----------+-----------+-------------+------------------+--------------+-----------+
|payment_type|n_paiements|n_commandes|valeur       |echeances_moyennes|part_paiements|part_valeur|
+------------+-----------+-----------+-------------+------------------+--------------+-----------+
|credit_card |75893      |75606      |1.234823505E7|3.51              |0.7401        |0.7848     |
|boleto      |19521      |19521      |2824521.49   |1.0               |0.1904        |0.1795     |
|voucher     |5613       |3745       |349874.4     |1.0               |0.0547        |0.0222     |
|debit_card  |1515       |1514       |212063.51    |1.0               |0.0148        |0.0135     |
+------------+-----------+-----------+-------------+------------------+--------------+-----------+

Somme des paiements : 102542 (doit égaler la ligne « Paiements des commandes is_sale »)

Commandes is_sale avec paiement : 9

In [43]:
cartes = cmp.filter("payment_types = 'credit_card'")
(cartes.withColumn("tranche_echeances", tr)
    .groupBy("tranche_echeances")
    .agg(F.count("*").alias("n_total"),
         F.sum((F.abs(F.col("ecart")) > 0.01).cast("int")).alias("n_ecart"),
         F.round(F.avg("ecart"), 3).alias("ecart_moyen_toutes_commandes"),
         F.round(F.avg(F.when(F.abs(F.col("ecart")) > 0.01, F.col("ecart"))), 2).alias("ecart_moyen_des_ecarts"))
    .withColumn("taux_ecart", F.round(F.col("n_ecart") / F.col("n_total"), 4))
    .orderBy("tranche_echeances").show(truncate=False))
print("Somme n_total :", cartes.count(), "(attendu 73756) ; somme n_ecart attendu 275")

+-----------------+-------+-------+----------------------------+----------------------+----------+
|tranche_echeances|n_total|n_ecart|ecart_moyen_toutes_commandes|ecart_moyen_des_ecarts|taux_ecart|
+-----------------+-------+-------+----------------------------+----------------------+----------+
|1_une_echeance   |23636  |9      |-0.001                      |-2.47                 |4.0E-4    |
|2_2_a_3          |22309  |52     |0.004                       |1.89                  |0.0023    |
|3_4_a_6          |15884  |106    |0.032                       |4.76                  |0.0067    |
|4_7_a_10         |11602  |84     |0.136                       |18.84                 |0.0072    |
|5_11_et_plus     |325    |24     |2.285                       |30.94                 |0.0738    |
+-----------------+-------+-------+----------------------------+----------------------+----------+

Somme n_total : 73756 (attendu 73756) ; somme n_ecart attendu 275


In [44]:
ecart_par_type = (ecarts.groupBy("payment_types")
    .agg(F.count("*").alias("n"),
         F.sum((F.col("ecart") > 0).cast("int")).alias("n_positifs"),
         F.sum((F.col("ecart") < 0).cast("int")).alias("n_negatifs"),
         F.round(F.avg("ecart"), 2).alias("ecart_moyen"),
         F.round(F.min("ecart"), 2).alias("ecart_min"),
         F.round(F.max("ecart"), 2).alias("ecart_max")))

ecrire_gold(par_type, "kpi_m6_type")
ecrire_gold(par_combo, "kpi_m6_combo")
ecrire_gold(par_echeances, "kpi_m6_echeances")
ecrire_gold(ecart_par_type, "kpi_m6_ecart_paiement")

t1 = spark.read.parquet(GOLD + "kpi_m6_type")
t2 = spark.read.parquet(GOLD + "kpi_m6_combo")
t3 = spark.read.parquet(GOLD + "kpi_m6_echeances")
t4 = spark.read.parquet(GOLD + "kpi_m6_ecart_paiement")
print("kpi_m6_type           : lignes", t1.count(), "(attendu 4), paiements =",
      t1.agg(F.sum("n_paiements")).first()[0], "(attendu 102542)")
print(f"  valeur              : {t1.agg(F.sum('valeur')).first()[0]:,.2f} (attendu 15 734 694,45)")
print("kpi_m6_combo          : lignes", t2.count(), "(attendu 6), commandes =",
      t2.agg(F.sum("n_commandes")).first()[0], "(attendu 98175)")
print("kpi_m6_echeances      : lignes", t3.count(), "(attendu 5), commandes =",
      t3.agg(F.sum("n_commandes")).first()[0], "(attendu 73395)")
print("kpi_m6_ecart_paiement : lignes", t4.count(), "(attendu 4), n =",
      t4.agg(F.sum("n")).first()[0], "(attendu 303)")

kpi_m6_type           : lignes 4 (attendu 4), paiements = 102542 (attendu 102542)
  valeur              : 15,734,694.45 (attendu 15 734 694,45)
kpi_m6_combo          : lignes 6 (attendu 6), commandes = 98175 (attendu 98175)
kpi_m6_echeances      : lignes 5 (attendu 5), commandes = 73395 (attendu 73395)
kpi_m6_ecart_paiement : lignes 4 (attendu 4), n = 303 (attendu 303)


In [45]:
import time, shutil, os

BENCH = GOLD + "_bench/"
foi_y = (spark.read.parquet(GOLD + "fact_order_items")
         .withColumn("order_year", F.floor(F.col("order_date_key") / 10000).cast("int")))
foi_y.groupBy("order_year").count().orderBy("order_year").show()

foi_y.coalesce(1).write.mode("overwrite").parquet(BENCH + "plat")
foi_y.write.mode("overwrite").partitionBy("order_year").parquet(BENCH + "partitionne")

def taille(chemin):
    fichiers = [os.path.join(r, f) for r, _, fs in os.walk(chemin) for f in fs if f.endswith(".parquet")]
    return len(fichiers), round(sum(os.path.getsize(f) for f in fichiers) / 1024, 1)
print("plat        : fichiers, Ko =", taille(BENCH + "plat"))
print("partitionné : fichiers, Ko =", taille(BENCH + "partitionne"))

plat = spark.read.parquet(BENCH + "plat")
part = spark.read.parquet(BENCH + "partitionne")

def mesurer(construire, n=5):
    construire().collect()                       # échauffement
    t = []
    for _ in range(n):
        d = time.perf_counter(); construire().collect(); t.append(time.perf_counter() - d)
    return round(sorted(t)[n // 2], 3)           # médiane en secondes

for nom, df in [("plat", plat), ("partitionné", part)]:
    print(f"{nom:12} CA 2018 : {df.filter('order_year = 2018').agg(F.sum('price')).first()[0]:,.2f}"
          f" | CA total : {df.agg(F.sum('price')).first()[0]:,.2f}")
print("(les CA doivent être identiques entre les deux versions)")

print("\nTemps médian en secondes (5 mesures après échauffement)")
for nom, df in [("plat", plat), ("partitionné", part)]:
    t_2018 = mesurer(lambda: df.filter("order_year = 2018").agg(F.sum("price")))
    t_tout = mesurer(lambda: df.agg(F.sum("price")))
    print(f"  {nom:12} filtre 2018 : {t_2018} | lecture totale : {t_tout}")

print("\nPlan, version partitionnée, filtre 2018")
part.filter("order_year = 2018").agg(F.sum("price")).explain()

shutil.rmtree(BENCH)
print("Dossier de test supprimé :", not os.path.exists(BENCH))

+----------+-----+
|order_year|count|
+----------+-----+
|      2016|  370|
|      2017|50846|
|      2018|61410|
+----------+-----+



plat        : fichiers, Ko = (1, 10070.1)
partitionné : fichiers, Ko = (3, 9684.1)
plat         CA 2018 : 7,385,097.92 | CA total : 13,588,545.09
partitionné  CA 2018 : 7,385,097.92 | CA total : 13,588,545.09
(les CA doivent être identiques entre les deux versions)

Temps médian en secondes (5 mesures après échauffement)
  plat         filtre 2018 : 0.09 | lecture totale : 0.074
  partitionné  filtre 2018 : 0.076 | lecture totale : 0.147

Plan, version partitionnée, filtre 2018
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[], functions=[sum(price#22066)])
   +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=36964]
      +- HashAggregate(keys=[], functions=[partial_sum(price#22066)])
         +- Project [price#22066]
            +- FileScan parquet [price#22066,order_year#22074] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/work/data/gold/_bench/partitionne], PartitionFilters: [isnotnull(order_y

In [46]:
BRONZE = "/home/work/data/bronze/"
TOL = 0.01

def somme(df, col):
    return df.agg(F.sum(col)).first()[0]

ca_bronze = somme(spark.read.parquet(BRONZE + "order_items").withColumn("p", F.col("price").cast("double")), "p")
foi = spark.read.parquet(GOLD + "fact_order_items")
fo = spark.read.parquet(GOLD + "fact_orders")
ca_quar = somme(lire_silver("quarantine_order_items"), "price")
ca_silver = somme(foi, "price")
ca_ventes = somme(foi.filter("is_sale"), "price")

lire_g = lambda nom: spark.read.parquet(GOLD + nom)
controles = [
    ("Bronze (source)",                         ca_bronze, 13591643.70),
    ("Quarantaine",                             ca_quar, 3098.61),
    ("Silver = fact_order_items",               ca_silver, 13588545.09),
    ("Commandes canceled/unavailable",          ca_silver - ca_ventes, 97242.96),
    ("CA des ventes (fact_order_items)",        ca_ventes, 13491302.13),
    ("CA des ventes (fact_orders)",             somme(fo.filter("is_sale"), "items_total"), 13491302.13),
    ("kpi_m1_pareto",                           somme(lire_g("kpi_m1_pareto"), "ca"), 13491302.13),
    ("kpi_m1_mensuel (fenêtre)",                somme(lire_g("kpi_m1_mensuel"), "ca"), 13446431.07),
    ("kpi_m4_rfm",                              somme(lire_g("kpi_m4_rfm"), "montant"), 13491302.13),
    ("kpi_m4_segment",                          somme(lire_g("kpi_m4_segment"), "ca"), 13491302.13),
    ("kpi_m5_statut (tous statuts)",            somme(lire_g("kpi_m5_statut"), "items_total"), 13588545.09),
]
ok = True
for nom, val, att in controles:
    ecart = round(val - att, 2)
    bon = abs(ecart) <= TOL
    ok &= bon
    print(f"{'OK' if bon else 'ÉCART':6} {nom:38} {val:>16,.2f}  attendu {att:>16,.2f}  écart {ecart}")
print("Identité Bronze - quarantaine - Silver :", round(ca_bronze - ca_quar - ca_silver, 2), "(attendu 0.0)")
print("Réconciliation globale :", "OK" if ok else "ÉCARTS À EXAMINER")

print("\nLignes des tables de base")
for nom, att in [("fact_order_items", 112626), ("fact_orders", 99410), ("dim_product", 32951),
                 ("dim_seller", 3095), ("dim_customer", 96067), ("dim_date", 852)]:
    print(f"  {nom:18} {lire_g(nom).count():>7} (attendu {att})")

dossiers = sorted(d for d in os.listdir(GOLD) if not d.startswith("_"))
print("\nTables dans data/gold :", len(dossiers), "(attendu 23 = 6 de base + 17 kpi)")
print(dossiers)

OK     Bronze (source)                           13,591,643.70  attendu    13,591,643.70  écart -0.0
OK     Quarantaine                                    3,098.61  attendu         3,098.61  écart 0.0
OK     Silver = fact_order_items                 13,588,545.09  attendu    13,588,545.09  écart 0.0
OK     Commandes canceled/unavailable                97,242.96  attendu        97,242.96  écart 0.0
OK     CA des ventes (fact_order_items)          13,491,302.13  attendu    13,491,302.13  écart 0.0
OK     CA des ventes (fact_orders)               13,491,302.13  attendu    13,491,302.13  écart 0.0
OK     kpi_m1_pareto                             13,491,302.13  attendu    13,491,302.13  écart 0.0
OK     kpi_m1_mensuel (fenêtre)                  13,446,431.07  attendu    13,446,431.07  écart 0.0
OK     kpi_m4_rfm                                13,491,302.13  attendu    13,491,302.13  écart 0.0
OK     kpi_m4_segment                            13,491,302.13  attendu    13,491,302.13  écart -0.

26/11/09 20:00:38 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/blockmgr-9508626c-2f2e-4d3a-97f0-b329bbfc4a85. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/blockmgr-9508626c-2f2e-4d3a-97f0-b329bbfc4a85
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:173)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:109)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:90)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1126)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1(DiskBlockManager.scala:368)
	at org.apache.spark.storage.DiskBlockManager.$anonfun$doStop$1$adapted(DiskBlockManager.scala:364)
	at scala.collection.IndexedSeqOptimize